# Storyline reproduction notebook: result panels only

This notebook reproduces and displays the manuscript result panels in the order of the paper's evidence chain. It deliberately excludes the Figure 1 architecture schematic, the methods flowchart and composite figure pages. Each result panel is shown in its own code cell so that an independent reader can inspect the table, rerun the plot and follow the logic one step at a time.

## 0. Set paths and runtime switches

The first cells point the notebook to the code repository, the deposited data package and a working directory. By default, model training is skipped and the deposited trained artifacts are used; set `RUN_MODEL_TRAINING = True` to retrain the two main models from the deposited training matrix. The manuscript-analysis pipeline is enabled, but it reuses existing outputs unless `FORCE_RERUN_MANUSCRIPT_PIPELINE = True`.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle


def find_code_repo() -> Path:
    env_path = os.environ.get('RNA_PORTRAIT_CODE_REPO')
    if env_path:
        candidate = Path(env_path).expanduser().resolve()
        if (candidate / 'rna_portrait').exists() and (candidate / 'workflows').exists():
            return candidate
        raise FileNotFoundError(f'RNA_PORTRAIT_CODE_REPO does not look like the code repository: {candidate}')
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        candidate = candidate.resolve()
        if (candidate / 'rna_portrait').exists() and (candidate / 'workflows').exists():
            return candidate
    raise FileNotFoundError('Could not infer CODE_REPO. Set RNA_PORTRAIT_CODE_REPO to the code repository path.')


CODE_REPO = find_code_repo()
DATA_PACKAGE = Path(os.environ.get('RNA_PORTRAIT_DATA_PACKAGE', CODE_REPO.parent / 'released_data_package')).expanduser().resolve()
WORK_DIR = Path(os.environ.get('RNA_PORTRAIT_WORK_DIR', CODE_REPO.parent / 'rna_portrait_reproduction_run')).expanduser().resolve()

RUN_MODEL_TRAINING = False
RUN_MANUSCRIPT_PIPELINE = True
FORCE_RERUN_MANUSCRIPT_PIPELINE = False
RUN_R_DECONVOLUTION = True
SAVE_PANEL_FILES = False

WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'logs').mkdir(exist_ok=True)
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print('CODE_REPO:', CODE_REPO)
print('DATA_PACKAGE:', DATA_PACKAGE)
print('WORK_DIR:', WORK_DIR)
print('RUN_MODEL_TRAINING:', RUN_MODEL_TRAINING)
print('RUN_MANUSCRIPT_PIPELINE:', RUN_MANUSCRIPT_PIPELINE)
print('SAVE_PANEL_FILES:', SAVE_PANEL_FILES)

## 1. Confirm that the required data and code are present

In [ ]:
required_code_files = [
    CODE_REPO / 'rna_portrait' / '__init__.py',
    CODE_REPO / 'workflows' / 'run_training_pipeline.py',
    CODE_REPO / 'workflows' / 'run_manuscript_pipeline.py',
    CODE_REPO / 'workflows' / 'figures' / 'make_main_figures_and_source_data.py',
]
required_data_paths = [
    DATA_PACKAGE / 'processed_data' / 'training_matrix' / 'expr_log.parquet',
    DATA_PACKAGE / 'processed_data' / 'training_matrix' / 'meta.csv',
    DATA_PACKAGE / 'processed_data' / 'training_matrix' / 'genes.npy',
    DATA_PACKAGE / 'processed_data' / 'training_matrix' / 'sample_ids.npy',
    DATA_PACKAGE / 'processed_data' / 'validation_profiles' / 'external_180',
    DATA_PACKAGE / 'processed_data' / 'validation_profiles' / 'multisource_450',
    DATA_PACKAGE / 'model_artifacts' / 'bulk_multimodal_embedding' / 'semantic_alignment_backbone' / 'bulk_multimodal_embedding.pt',
    DATA_PACKAGE / 'model_artifacts' / 'bulk_multimodal_embedding' / 'semantic_alignment_backbone' / 'sample_embeddings.csv',
    DATA_PACKAGE / 'model_artifacts' / 'bulk_multimodal_embedding' / 'semantic_backbone_v8_topk64' / 'semantic_prototype_attention.pt',
]
missing = [str(p) for p in required_code_files + required_data_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files or directories:' + chr(10) + chr(10).join(missing))
print('All required code and data paths are present.')

## 2. Load the deposited training matrix

This step shows what the RNA matrix and metadata look like before any manuscript result is plotted.

In [ ]:
TRAINING_MATRIX = DATA_PACKAGE / 'processed_data' / 'training_matrix'
meta = pd.read_csv(TRAINING_MATRIX / 'meta.csv')
genes = np.load(TRAINING_MATRIX / 'genes.npy', allow_pickle=True)
sample_ids = np.load(TRAINING_MATRIX / 'sample_ids.npy', allow_pickle=True)
summary = json.loads((TRAINING_MATRIX / 'summary.json').read_text())

print('training matrix samples:', len(sample_ids))
print('training matrix genes:', len(genes))
print('metadata shape:', meta.shape)
print('summary keys:', sorted(summary.keys()))
display(meta.head())

preview_sample_ids = sample_ids[:5].tolist()
expr_preview = pd.read_parquet(TRAINING_MATRIX / 'expr_log.parquet', columns=preview_sample_ids).head(8)
print('Expression preview shape:', expr_preview.shape)
display(expr_preview)

## 3. Optional model training

In [ ]:
TRAINED_ARTIFACT_ROOT = WORK_DIR / 'trained_artifacts'
training_cmd = [
    sys.executable, str(CODE_REPO / 'workflows' / 'run_training_pipeline.py'),
    '--training-matrix', str(TRAINING_MATRIX),
    '--artifact-root', str(TRAINED_ARTIFACT_ROOT),
    '--alignment-run-name', 'semantic_alignment_backbone',
    '--portrait-run-name', 'semantic_backbone_v8_topk64',
    '--skip-age-readout',
]
print(' '.join(training_cmd))
trained_summary_path = TRAINED_ARTIFACT_ROOT / 'bulk_multimodal_embedding' / 'semantic_backbone_v8_topk64' / 'summary.json'
if RUN_MODEL_TRAINING and trained_summary_path.exists():
    print('Training artifacts already exist; skipping retraining. Delete', TRAINED_ARTIFACT_ROOT, 'to rerun from scratch.')
elif RUN_MODEL_TRAINING:
    log_path = WORK_DIR / 'logs' / 'training_pipeline.log'
    env = os.environ.copy()
    env.setdefault('TOKENIZERS_PARALLELISM', 'false')
    with log_path.open('w', encoding='utf-8') as log:
        subprocess.run(training_cmd, cwd=str(CODE_REPO), stdout=log, stderr=subprocess.STDOUT, check=True, env=env)
    print('Training finished. Log:', log_path)
else:
    print('Training skipped. Set RUN_MODEL_TRAINING = True in the setup cell to retrain from the deposited training matrix.')

## 4. Regenerate or reuse manuscript analysis tables

The downstream panels read either regenerated source-data tables under `WORK_DIR/figures/source_data` or regenerated analysis tables under `WORK_DIR/manuscript_analysis_tables`.

In [ ]:
ANALYSIS_OUT = WORK_DIR / 'manuscript_analysis_tables'
FIGURE_OUT = WORK_DIR / 'figures'
R_LIB_ROOT = DATA_PACKAGE / 'software_environment' / 'r_libraries_deconvolution'

manuscript_cmd = [
    sys.executable, str(CODE_REPO / 'workflows' / 'run_manuscript_pipeline.py'),
    '--data-root', str(DATA_PACKAGE / 'processed_data'),
    '--artifact-root', str(DATA_PACKAGE / 'model_artifacts'),
    '--analysis-output-root', str(ANALYSIS_OUT),
    '--figure-output-root', str(FIGURE_OUT),
]
if RUN_R_DECONVOLUTION:
    manuscript_cmd += ['--r-lib-root', str(R_LIB_ROOT)]
else:
    manuscript_cmd += ['--skip-r-deconvolution']
print(' '.join(manuscript_cmd))

required_pipeline_outputs = [
    ANALYSIS_OUT / 'T1_RNA_language_alignment' / 't1_cosine_summary.csv',
    ANALYSIS_OUT / 'T13_portrait_composition_disease_information' / 't13_model_performance_summary.csv',
    FIGURE_OUT / 'source_data' / 'figure_2_cosine_summary.csv',
    FIGURE_OUT / 'source_data' / 'extended_data_1_cv_comparison.csv',
]
outputs_present = all(p.exists() for p in required_pipeline_outputs)

if RUN_MANUSCRIPT_PIPELINE and (FORCE_RERUN_MANUSCRIPT_PIPELINE or not outputs_present):
    log_path = WORK_DIR / 'logs' / 'manuscript_pipeline.log'
    with log_path.open('w', encoding='utf-8') as log:
        subprocess.run(manuscript_cmd, cwd=str(CODE_REPO), stdout=log, stderr=subprocess.STDOUT, check=True)
    print('Manuscript pipeline finished. Log:', log_path)
elif RUN_MANUSCRIPT_PIPELINE and outputs_present:
    print('Pipeline outputs already exist and will be reused. Set FORCE_RERUN_MANUSCRIPT_PIPELINE = True to recompute them.')
else:
    print('Manuscript pipeline skipped. The notebook will use existing analysis/source-data outputs if present.')

SOURCE_DIR = FIGURE_OUT / 'source_data'
if not SOURCE_DIR.exists():
    fallback = DATA_PACKAGE / 'source_data'
    if fallback.exists():
        SOURCE_DIR = fallback
    else:
        raise FileNotFoundError('No source-data directory found. Run the manuscript pipeline first.')

RESULTS = ANALYSIS_OUT
OUT_DIR = WORK_DIR / 'storyline_panel_exports'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Analysis tables:', RESULTS)
print('Source-data tables:', SOURCE_DIR)
print('Panel export directory:', OUT_DIR)

## 5. Shared plotting helpers

These helpers define the reader-facing portrait labels, common fonts and the `read_table()` function used by every panel below. Raw result-table keys such as `stable_consensus` are mapped to manuscript-facing labels such as `single clear signal`.

In [ ]:
PALETTE = {
    'blue': '#1F5A9D',
    'blue_light': '#9DB9D8',
    'blue_soft': '#DCE8F3',
    'red': '#B64A4A',
    'red_light': '#E7B1AA',
    'green': '#5B8C5A',
    'green_light': '#B8D3B4',
    'violet': '#7E6AAE',
    'orange': '#C9822B',
    'orange_light': '#E1B66D',
    'grey': '#8F8F8F',
    'grey_light': '#D9D9D9',
    'black': '#222222',
}

STATE_ORDER = [
    'stable_consensus',
    'hematologic_override',
    'epithelial_override',
    'clean_anchor_override',
    'generic_context_override',
    'unsupported_semantics',
    'family_conflict',
    'other',
]

STATE_LABELS = {
    'stable_consensus': 'single clear signal',
    'hematologic_override': 'blood/immune',
    'epithelial_override': 'epithelial-like context',
    'clean_anchor_override': 'cleaner anchor context',
    'generic_context_override': 'broad context',
    'unsupported_semantics': 'weak evidence',
    'family_conflict': 'conflict',
    'other': 'other',
}

STATUS_LABELS = {
    'single_clear_signal': 'single clear signal',
    'mixed': 'several signals',
    'unsupported': 'weak evidence',
}
STATUS_KEY_MAP = {
    'stable': 'single_clear_signal',  # raw result-table key; reader label is single clear signal
    'mixed': 'mixed',
    'unsupported': 'unsupported',
}

STATE_COLORS = {
    'stable_consensus': '#303030',
    'hematologic_override': PALETTE['blue'],
    'epithelial_override': PALETTE['red'],
    'clean_anchor_override': PALETTE['green'],
    'generic_context_override': PALETTE['violet'],
    'unsupported_semantics': '#A9A9A9',
    'family_conflict': PALETTE['orange'],
    'other': '#6B6B6B',
}

TITLE_FONTSIZE = 9
LABEL_FONTSIZE = 8
TICK_FONTSIZE = 8
LEGEND_FONTSIZE = 8

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'font.size': LABEL_FONTSIZE,
    'axes.labelsize': LABEL_FONTSIZE,
    'axes.titlesize': TITLE_FONTSIZE,
    'xtick.labelsize': TICK_FONTSIZE,
    'ytick.labelsize': TICK_FONTSIZE,
    'legend.fontsize': LEGEND_FONTSIZE,
    'axes.linewidth': 0.7,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
    'figure.max_open_warning': 100,
})

def apply_font_sizes(fig_or_ax, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE):
    """Apply explicit font sizes to titles, labels, ticks, legends and colorbars."""
    axes = fig_or_ax.axes if hasattr(fig_or_ax, 'axes') else [fig_or_ax]
    for ax in axes:
        ax.title.set_fontsize(title_fontsize)
        ax.xaxis.label.set_size(label_fontsize)
        ax.yaxis.label.set_size(label_fontsize)
        ax.tick_params(axis='both', labelsize=tick_fontsize)
        legend = ax.get_legend()
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontsize(legend_fontsize)
            title = legend.get_title()
            if title is not None:
                title.set_fontsize(legend_fontsize)
        # Colorbar axes do not always expose labels through the parent plot.
        for text in ax.texts:
            if text.get_fontsize() < label_fontsize:
                text.set_fontsize(label_fontsize)
    return fig_or_ax


TABLES = {
    's1_cv': 'T12_whole_profile_vs_local_parts/t12_cv_model_comparison.csv',
    's1_reducibility': 'T12_whole_profile_vs_local_parts/t12_portrait_reducibility.csv',
    's2_external_meta': 'T4d_official_EPIC_deconvolution/external_180_metadata.csv',
    's2_multisource_meta': 'T4d_official_EPIC_deconvolution/multisource_450_metadata.csv',
    's2_quality': 'T10_quality_heterogeneity_robustness/t10_quality_by_portrait.csv',
    's3_cosine_controls': 'T1_RNA_language_alignment/t1_cosine_controls.csv',
    's3_exact': 'T1_RNA_language_alignment/t1_exact_retrieval_metrics.csv',
    's3_broad': 'T1_RNA_language_alignment/t1_broad_semantic_retrieval.csv',
    's4_reliability': 'T2_calibrated_closed_set/t2b_internal_reliability_bins.csv',
    's4_risk_coverage': 'T2_calibrated_closed_set/t2b_internal_risk_coverage.csv',
    's4_external_thresholds': 'T2_calibrated_closed_set/t2b_external_thresholds.csv',
    's4_external_predictions': 'T2_calibrated_closed_set/t2b_external_predictions.csv',
    's5_status_transition': 'sample_level_disease_transition/raw_to_resolved_status.csv',
    's5_family_transition': 'sample_level_disease_transition/raw_to_resolved_family.csv',
    's5_expected_transition': 'sample_level_disease_transition/sample_level_transitions.csv',
    's6_marker': 'T4_marker_program_state_validation/t4_marker_scores_by_state.csv',
    's6_pathway': 'T4b_pathway_state_validation/t4b_pathway_scores_by_state.csv',
    's6_epic': 'T4d_official_EPIC_deconvolution/t4d_epic_state_vs_rest_effects.csv',
    's6_mcp': 'T4e_official_MCPcounter_deconvolution/t4e_mcpcounter_state_vs_rest_effects.csv',
    's7_mixing': 'T5c_expanded_calibrated_mixing_bootstrap/t5c_fraction_summary.csv',
    's8_metadata': 'T8_shortcut_exclusion_controls/t8_shortcut_association.csv',
    's8_partial_r2': 'T9_strict_shortcut_residual_controls/t9_partial_r2_after_controls.csv',
    's8_within_label': 'T9_strict_shortcut_residual_controls/t9_within_label_evidence_separation_summary.csv',
    's9_robustness': 'T10_quality_heterogeneity_robustness/t10_subset_robustness_summary.csv',
    's10_boundaries': 'T11_failure_mode_reliability_boundaries/t11_boundary_flag_summary.csv',
    's10_reliability': 'T11_failure_mode_reliability_boundaries/t11_reliability_by_portrait.csv',
    's11_composition_long': 'T13_portrait_composition_disease_information/t13_disease_portrait_composition_long.csv',
    's11_model_performance': 'T13_portrait_composition_disease_information/t13_model_performance_summary.csv',
    's11_permutation': 'T13_portrait_composition_disease_information/t13_within_tissue_source_permutation.csv',
    's11_group_identification': 'T13_portrait_composition_disease_information/t13_group_level_composition_identification_by_disease.csv',
}


def read_table(key):
    source_path = SOURCE_DIR / f'{key}.csv'
    if source_path.exists():
        return pd.read_csv(source_path)
    if key not in TABLES:
        raise KeyError(f'Unknown table key: {key}')
    path = RESULTS / TABLES[key]
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def clean_axis(ax, grid='y'):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if grid:
        ax.grid(axis=grid, color='#E8E8E8', lw=0.55, zorder=0)
    ax.set_axisbelow(True)


def wrap_label(text, width=14):
    return '\n'.join(textwrap.wrap(str(text), width=width, break_long_words=False))


def portrait_display_label(value):
    aliases = {
        'stable': STATE_LABELS['stable_consensus'],  # raw result-table key only
        'epithelial': STATE_LABELS['epithelial_override'],
        'clean anchor': STATE_LABELS['clean_anchor_override'],
    }
    text = str(value)
    return aliases.get(text, STATE_LABELS.get(text, text))


def portrait_display_labels(values, width=22):
    return [wrap_label(portrait_display_label(value), width) for value in values]


def save_panel(fig, name, dpi_png=180, dpi_tiff=300):
    paths = {}
    if SAVE_PANEL_FILES:
        for ext in ['svg', 'pdf', 'png', 'tiff']:
            path = OUT_DIR / f'{name}.{ext}'
            kwargs = {'bbox_inches': 'tight', 'pad_inches': 0.04}
            if ext == 'png':
                kwargs['dpi'] = dpi_png
            if ext == 'tiff':
                kwargs['dpi'] = dpi_tiff
            fig.savefig(path, **kwargs)
            paths[ext] = path
        print('saved:', paths['svg'])
    display(fig)
    plt.close(fig)
    return paths


def row_normalized_pivot(df, index, columns, values='n', top_n=None):
    table = df.pivot_table(index=index, columns=columns, values=values, aggfunc='sum', fill_value=0)
    if top_n:
        table = table.loc[table.sum(axis=1).sort_values(ascending=False).head(top_n).index]
    return table.div(table.sum(axis=1), axis=0).fillna(0)

CLAIM_LABELS = {
    'epithelial_or_tumor_like_signal': 'epithelial or tumour-like signal',
    'clean_or_non_malignant_context': 'cleaner or non-malignant context',
    'mixed_or_unstable_disease_reading': 'mixed or inconsistent label reading',
    'immune_or_blood_signal': 'immune or blood signal',
    'context_or_stromal_signal': 'stromal or broader-context signal',
}
CLAIM_TYPE_LABELS = {
    'epithelial_or_tumor_like_signal': 'epithelial_or_tumour_like_signal',
    'clean_or_non_malignant_context': 'cleaner_or_non_malignant_context',
    'mixed_or_unstable_disease_reading': 'mixed_or_inconsistent_label_reading',
    'immune_or_blood_signal': 'immune_or_blood_signal',
    'context_or_stromal_signal': 'stromal_or_broader_context_signal',
}


# Evidence block 1. What data are being read?

Purpose: document the training representation and external cohort heterogeneity before interpreting model outputs. These are Supplementary panels because they are data-context and quality-control views.

In [ ]:
# S1 input. Trained RNA embedding PCA coordinates for all training/validation/test profiles.
training_embeddings_path = (
    DATA_PACKAGE
    / 'model_artifacts'
    / 'bulk_multimodal_embedding'
    / 'semantic_alignment_backbone'
    / 'sample_embeddings.csv'
)
training_embeddings = pd.read_csv(training_embeddings_path)
print('training embedding table:', training_embeddings.shape)
display(training_embeddings[['sample_id', 'gse', 'split', 'feat_anatomical_site', 'feat_tumor_status', 'feat_disease_label', 'pca1', 'pca2', 'expr_text_gap']].head())

**Caption (English).** Source table for Supplementary Fig. S1. The table contains one row per training-set-related RNA profile, including the trained RNA embedding PCA coordinates used below.

**图注（中文）.** Supplementary Fig. S1 的来源表。每一行是一份训练相关 RNA 样本，包含下方散点图使用的训练后 RNA embedding PCA 坐标。

**小朋友版解释.** 这里先打开数据表，确认每个样本都有名字、来源、标签和两个 PCA 坐标。后面的图就是把这些点画到同一张地图上。


## Supplementary Fig. 1a. Training RNA embedding space by split

In [ ]:
# S1a. Training RNA embedding PCA, coloured by data split.
data = training_embeddings.copy()
order = ['train', 'val', 'test']
colors = {'train': PALETTE['grey_light'], 'val': PALETTE['blue_light'], 'test': PALETTE['blue']}

fig, ax = plt.subplots(figsize=(4.0, 3.4))
for split in order:
    sub = data[data['split'] == split]
    ax.scatter(sub['pca1'], sub['pca2'], s=3.0, alpha=0.42, color=colors[split], label=f'{split} (n={len(sub):,})', linewidths=0, rasterized=False)
ax.set_xlabel('RNA embedding PC1')
ax.set_ylabel('RNA embedding PC2')
ax.set_title('Training RNA embedding space by split')
ax.legend(loc='best', markerscale=4)
clean_axis(ax, grid=None)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s1a_paths = save_panel(fig, 'S1a_training_embedding_pca_by_split')


**Caption (English).** PCA projection of 25,199 trained RNA embeddings, coloured by train, validation and test split. Each dot is one RNA profile; the x and y axes are the first two displayed principal components of the trained RNA embedding. This panel checks that the embedding space is not collapsed and that split assignment does not create a separate visual island.

**图注（中文）.** 25,199 个训练后 RNA embedding 的 PCA 投影，按训练集、验证集和测试集上色。每个点是一份 RNA 样本；横轴和纵轴是训练后 RNA embedding 的前两个展示用主成分。该图用于检查表征空间没有塌缩，并观察数据划分是否形成明显孤岛。

**小朋友版解释.** 每个点是一份 RNA。颜色表示它被分到训练、验证还是测试。三种颜色如果大体混在同一张地图里，说明不是某一组数据单独跑到另一个世界去了。


## Supplementary Fig. 1b. Training RNA embedding space by tissue/site

In [ ]:
# S1b. Training RNA embedding PCA, coloured by anatomical site.
data = training_embeddings.copy()
data['site_plot'] = data['feat_anatomical_site'].fillna('unknown').astype(str)
top_sites = data['site_plot'].value_counts().head(10).index.tolist()
data['site_plot'] = np.where(data['site_plot'].isin(top_sites), data['site_plot'], 'other sites')
category_order = ['other sites'] + top_sites
category_order = [c for c in category_order if c in set(data['site_plot'])]
color_list = [PALETTE['grey_light'], PALETTE['blue'], PALETTE['red'], PALETTE['green'], PALETTE['orange'], PALETTE['violet'], '#4B8B8C', '#A65E7E', '#7A7A45', '#6D88B8', '#B08A4A']
color_map = dict(zip(category_order, color_list))

fig, ax = plt.subplots(figsize=(4.4, 3.6))
for site in category_order:
    sub = data[data['site_plot'] == site]
    alpha = 0.18 if site == 'other sites' else 0.48
    size = 2.4 if site == 'other sites' else 3.2
    ax.scatter(sub['pca1'], sub['pca2'], s=size, alpha=alpha, color=color_map[site], label=f'{site.replace("_", " ")} (n={len(sub):,})', linewidths=0, rasterized=False)
ax.set_xlabel('RNA embedding PC1')
ax.set_ylabel('RNA embedding PC2')
ax.set_title('Training RNA embedding space by tissue/site')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), markerscale=4, borderaxespad=0)
clean_axis(ax, grid=None)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s1b_paths = save_panel(fig, 'S1b_training_embedding_pca_by_site')


**Caption (English).** The same trained RNA embedding PCA projection coloured by anatomical site. Colours mark the ten most frequent site labels, with remaining sites grouped as other sites. This panel documents how tissue/site structure contributes to the training representation.

**图注（中文）.** 同一训练后 RNA embedding PCA 投影，按解剖部位上色。颜色标出数量最多的十个部位标签，其余合并为 other sites。该图用于记录组织/部位结构如何影响训练表征空间。

**小朋友版解释.** 这还是同一张地图，只是这次用颜色标出样本来自身体哪里。如果某些颜色聚在一起，说明来自相似部位的 RNA 也更像。


## Supplementary Fig. 1c. Training RNA embedding space by tumour status

In [ ]:
# S1c. Training RNA embedding PCA, coloured by tumour status.
data = training_embeddings.copy()
data['tumour_status_plot'] = data['feat_tumor_status'].fillna('unknown').astype(str)
order = ['tumor', 'non_tumor', 'metastatic', 'adjacent_normal', 'not_applicable', 'unknown']
order = [x for x in order if x in set(data['tumour_status_plot'])]
color_map = {
    'tumor': PALETTE['red'],
    'non_tumor': PALETTE['green'],
    'metastatic': PALETTE['orange'],
    'adjacent_normal': PALETTE['blue_light'],
    'not_applicable': PALETTE['grey_light'],
    'unknown': PALETTE['grey'],
}

fig, ax = plt.subplots(figsize=(4.0, 3.4))
for status in order:
    sub = data[data['tumour_status_plot'] == status]
    ax.scatter(sub['pca1'], sub['pca2'], s=3.0, alpha=0.42, color=color_map.get(status, PALETTE['grey']), label=f'{status.replace("_", " ")} (n={len(sub):,})', linewidths=0, rasterized=False)
ax.set_xlabel('RNA embedding PC1')
ax.set_ylabel('RNA embedding PC2')
ax.set_title('Training RNA embedding space by tumour status')
ax.legend(loc='best', markerscale=4)
clean_axis(ax, grid=None)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s1c_paths = save_panel(fig, 'S1c_training_embedding_pca_by_tumour_status')


**Caption (English).** Trained RNA embedding PCA projection coloured by tumour status. The panel shows that tumour-related metadata is a major axis of structure in the training representation and therefore must be considered when interpreting downstream portraits.

**图注（中文）.** 训练后 RNA embedding PCA 投影，按肿瘤状态上色。该图显示肿瘤相关 metadata 是训练表征空间中的重要结构，因此解释后续分子画像时必须考虑这一因素。

**小朋友版解释.** 这张图用颜色标出样本是不是肿瘤。某些颜色如果集中在一起，说明模型地图里确实有肿瘤状态相关的方向。


## Supplementary Fig. 1d. Training RNA embedding space by disease label

In [ ]:
# S1d. Training RNA embedding PCA, coloured by frequent disease labels.
data = training_embeddings.copy()
data['disease_plot'] = data['feat_disease_label'].fillna('unknown').astype(str)
top_diseases = data['disease_plot'].value_counts().head(10).index.tolist()
data['disease_plot'] = np.where(data['disease_plot'].isin(top_diseases), data['disease_plot'], 'other disease labels')
category_order = ['other disease labels'] + top_diseases
category_order = [c for c in category_order if c in set(data['disease_plot'])]
color_list = [PALETTE['grey_light'], PALETTE['blue'], PALETTE['red'], PALETTE['green'], PALETTE['orange'], PALETTE['violet'], '#4B8B8C', '#A65E7E', '#7A7A45', '#6D88B8', '#B08A4A']
color_map = dict(zip(category_order, color_list))

fig, ax = plt.subplots(figsize=(4.6, 3.6))
for disease in category_order:
    sub = data[data['disease_plot'] == disease]
    alpha = 0.16 if disease == 'other disease labels' else 0.48
    size = 2.3 if disease == 'other disease labels' else 3.2
    ax.scatter(sub['pca1'], sub['pca2'], s=size, alpha=alpha, color=color_map[disease], label=f'{wrap_label(disease, 26)} (n={len(sub):,})', linewidths=0, rasterized=False)
ax.set_xlabel('RNA embedding PC1')
ax.set_ylabel('RNA embedding PC2')
ax.set_title('Training RNA embedding space by disease label')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), markerscale=4, borderaxespad=0)
clean_axis(ax, grid=None)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s1d_paths = save_panel(fig, 'S1d_training_embedding_pca_by_disease_label')


**Caption (English).** Trained RNA embedding PCA projection coloured by the ten most frequent disease labels. This is a dataset-context view: visible clusters can reflect disease biology, tissue composition and source structure, so this panel is not used as stand-alone evidence for external generalization.

**图注（中文）.** 训练后 RNA embedding PCA 投影，按数量最多的十个疾病标签上色。该图是数据背景图：可见聚类可能同时反映疾病生物学、组织组成和数据来源结构，因此不能单独作为外部泛化证据。

**小朋友版解释.** 颜色表示常见疾病名字。看到同色点靠近，说明这些样本在模型地图里相似；但这不一定只因为疾病，也可能因为它们来自相似组织或相似数据来源。


## Supplementary Fig. 1e. Training RNA embedding space by source

In [ ]:
# S1e. Training RNA embedding PCA, coloured by source dataset/project.
data = training_embeddings.copy()
data['source_plot'] = data['gse'].fillna('unknown').astype(str)
top_sources = data['source_plot'].value_counts().head(12).index.tolist()
data['source_plot'] = np.where(data['source_plot'].isin(top_sources), data['source_plot'], 'other datasets')
category_order = ['other datasets'] + top_sources
category_order = [c for c in category_order if c in set(data['source_plot'])]
color_list = [PALETTE['grey_light'], PALETTE['blue'], PALETTE['red'], PALETTE['green'], PALETTE['orange'], PALETTE['violet'], '#4B8B8C', '#A65E7E', '#7A7A45', '#6D88B8', '#B08A4A', '#8A6F4D', '#577590']
color_map = dict(zip(category_order, color_list))

fig, ax = plt.subplots(figsize=(4.8, 3.7))
for source in category_order:
    sub = data[data['source_plot'] == source]
    alpha = 0.14 if source == 'other datasets' else 0.48
    size = 2.2 if source == 'other datasets' else 3.2
    ax.scatter(sub['pca1'], sub['pca2'], s=size, alpha=alpha, color=color_map[source], label=f'{wrap_label(source, 22)} (n={len(sub):,})', linewidths=0, rasterized=False)
ax.set_xlabel('RNA embedding PC1')
ax.set_ylabel('RNA embedding PC2')
ax.set_title('Training RNA embedding space by source')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), markerscale=4, borderaxespad=0)
clean_axis(ax, grid=None)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s1e_paths = save_panel(fig, 'S1e_training_embedding_pca_by_source')


**Caption (English).** Trained RNA embedding PCA projection coloured by source dataset or project. Source structure is expected in a large public compendium and motivates the downstream source-control analyses rather than replacing them.

**图注（中文）.** 训练后 RNA embedding PCA 投影，按数据集或项目来源上色。在大型公共数据汇总中出现来源结构是预期现象，因此该图支持后续进行来源控制分析，而不能替代这些控制分析。

**小朋友版解释.** 这张图看样本来自哪个数据项目。颜色如果分成一团一团，说明“从哪里来的”也会影响地图位置，所以后面要专门检查模型是不是只学了来源信息。


## Supplementary Fig. 1f. RNA-text gap across the training embedding space

In [ ]:
# S1f. Training RNA embedding PCA, coloured by RNA-text gap.
data = training_embeddings.copy()
fig, ax = plt.subplots(figsize=(4.2, 3.4))
scatter = ax.scatter(
    data['pca1'],
    data['pca2'],
    s=3.0,
    alpha=0.55,
    c=data['expr_text_gap'],
    cmap=LinearSegmentedColormap.from_list('gap', [PALETTE['blue'], '#F6F6F6', PALETTE['red']]),
    vmin=float(data['expr_text_gap'].quantile(0.02)),
    vmax=float(data['expr_text_gap'].quantile(0.98)),
    linewidths=0,
    rasterized=False,
)
ax.set_xlabel('RNA embedding PC1')
ax.set_ylabel('RNA embedding PC2')
ax.set_title('RNA-text gap across training space')
fig.colorbar(scatter, ax=ax, label='RNA-text gap')
clean_axis(ax, grid=None)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s1f_paths = save_panel(fig, 'S1f_training_embedding_pca_by_rna_text_gap')


**Caption (English).** Trained RNA embedding PCA projection coloured by RNA-text gap. The colour scale reports how far each RNA embedding lies from its paired text embedding in the aligned space; warmer colours indicate larger gaps and therefore samples that may be less well aligned.

**图注（中文）.** 训练后 RNA embedding PCA 投影，按 RNA-text gap 连续值上色。颜色表示每个 RNA embedding 与其配对文本 embedding 在对齐空间中的距离；颜色越暖，gap 越大，提示该样本的 RNA-文字对齐可能较弱。

**小朋友版解释.** 这张图给每个点涂上“RNA 和文字配得多好”的颜色。蓝色更像配得近，红色更像离得远。它帮助我们找哪些区域可能更难被文字描述清楚。
<!-- RNA_PORTRAIT_SUPP_TRAINING_LANDSCAPE_END -->


## Supplementary Fig. 2a. External tissue/site composition

In [ ]:
# S2a. External sample composition by tissue/site family.
external = read_table('s2_external_meta')
multi = read_table('s2_multisource_meta')
data = pd.concat([external, multi], ignore_index=True)
counts = data.groupby(['pool', 'expected_site_family']).size().reset_index(name='n')
top_sites = counts.groupby('expected_site_family')['n'].sum().sort_values(ascending=False).head(9).index
plot = counts[counts['expected_site_family'].isin(top_sites)].pivot(index='expected_site_family', columns='pool', values='n').fillna(0)
plot = plot.loc[plot.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(4.0, 3.2))
y = np.arange(len(plot))
left = np.zeros(len(plot))
for pool, color in zip(plot.columns, [PALETTE['blue_light'], PALETTE['blue']]):
    ax.barh(y, plot[pool], left=left, color=color, label=pool)
    left += plot[pool].to_numpy()
ax.set_yticks(y, [wrap_label(x, 18) for x in plot.index])
ax.set_xlabel('number of external profiles')
ax.set_title('External data composition by tissue/site')
ax.legend(loc='best')
clean_axis(ax, grid='x')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s2a_paths = save_panel(fig, 'S2a_external_tissue_site_composition')

**Caption (English).** Composition of external profiles by tissue or site family. The x axis shows the number of profiles and the y axis shows major tissue/site families. Colours separate the two external pools.

**图注（中文）.** 外部样本按组织或部位分类的组成。横坐标是样本数量，纵坐标是主要组织/部位类别，颜色区分两个外部数据集。

**小朋友版解释.** 这张图是在数样本来自身体哪里。横向条越长，说明这种组织来源的样本越多。

## Supplementary Fig. 2b. External expected disease composition

In [ ]:
# S2b. External sample composition by expected disease family.
external = read_table('s2_external_meta')
multi = read_table('s2_multisource_meta')
data = pd.concat([external, multi], ignore_index=True)
counts = data.groupby(['pool', 'expected_disease_family']).size().reset_index(name='n')
top_diseases = counts.groupby('expected_disease_family')['n'].sum().sort_values(ascending=False).head(10).index
plot = counts[counts['expected_disease_family'].isin(top_diseases)].pivot(index='expected_disease_family', columns='pool', values='n').fillna(0)
plot = plot.loc[plot.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(4.2, 3.4))
y = np.arange(len(plot))
left = np.zeros(len(plot))
for pool, color in zip(plot.columns, [PALETTE['blue_light'], PALETTE['blue']]):
    ax.barh(y, plot[pool], left=left, color=color, label=pool)
    left += plot[pool].to_numpy()
ax.set_yticks(y, [wrap_label(x.replace('_', ' '), 20) for x in plot.index])
ax.set_xlabel('number of external profiles')
ax.set_title('External data composition by disease family')
ax.legend(loc='best')
clean_axis(ax, grid='x')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s2b_paths = save_panel(fig, 'S2b_external_disease_composition')

**Caption (English).** Composition of external profiles by expected disease family. The x axis shows the number of profiles and the y axis shows the main disease families represented in the external pools.

**图注（中文）.** 外部样本按预期疾病类别分类的组成。横坐标是样本数量，纵坐标是主要疾病类别，颜色区分两个外部数据集。

**小朋友版解释.** 这张图是在数样本大概属于哪类疾病。条越长，说明这种疾病样本越多。

## Supplementary Fig. 2c. Gene coverage by portrait group

In [ ]:
# S2c. Gene coverage by portrait group.
data = read_table('s2_quality').sort_values('median_gene_coverage')
fig, ax = plt.subplots(figsize=(4.0, 3.2))
y = np.arange(len(data))
median = data['median_gene_coverage'].to_numpy()
xerr = np.vstack([median - data['q25_gene_coverage'].to_numpy(), data['q75_gene_coverage'].to_numpy() - median])
ax.barh(y, median, color=PALETTE['blue_light'])
ax.errorbar(median, y, xerr=xerr, fmt='none', color=PALETTE['black'], capsize=2, lw=0.8)
ax.set_yticks(y, portrait_display_labels(data['label'], 22))
ax.set_xlabel('gene coverage')
ax.set_xlim(0, max(0.75, float(data['q75_gene_coverage'].max()) * 1.1))
ax.set_title('Gene coverage by portrait group')
clean_axis(ax, grid='x')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s2c_paths = save_panel(fig, 'S2c_gene_coverage_by_portrait')

**Caption (English).** Gene coverage by portrait group. The x axis shows the fraction of selected genes matched in each profile group; bars show medians and error bars show interquartile ranges.

**图注（中文）.** 不同画像组的基因覆盖度。横坐标是每组样本匹配到的基因比例，柱子表示中位数，误差线表示四分位范围。

**小朋友版解释.** 这张图看每类画像的“拼图块”够不够多。横向越长，说明这类样本里可用的基因信息越多。

# Evidence block 2. RNA profiles align with language

Claim tested here: paired RNA profiles and text descriptions are closer than shuffled controls, and broad semantic attributes can be recovered from RNA-text alignment.

## Main Fig. 2a. External RNA profile coordinates

In [ ]:
# Figure 2a. Standard scatter plot: external RNA profile coordinates.
data = read_table('figure_2_expression_coordinates')
fig, ax = plt.subplots(figsize=(4.4, 4.0))

for state in STATE_ORDER:
    sub = data[data['semantic_state_family'] == state]
    if len(sub) == 0:
        continue
    ax.scatter(
        sub['expr_x'], sub['expr_y'],
        s=16,
        alpha=0.75,
        color=STATE_COLORS[state],
        edgecolors='none',
        label=STATE_LABELS[state],
    )

ax.set_xlabel('RNA coordinate 1')
ax.set_ylabel('RNA coordinate 2')
ax.set_title('External RNA profiles')
ax.legend(loc='best', ncol=1, markerscale=0.9)
clean_axis(ax, grid=None)
fig2a_paths = save_panel(fig, 'Figure_2a_scatter_external_rna_profiles')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_2a_scatter_external_rna_profiles -->
**Caption (English).** Two-dimensional PCA projection of 630 external bulk RNA profiles. Each dot is one sample; the x and y axes are the first two displayed PCA coordinates computed from standardized log1p CPM expression values of the most variable common genes, and colours indicate the assigned molecular portrait family.

**图注（中文）。** 630 个外部 bulk RNA 样本的二维 PCA 投影。每个点代表一个样本；横轴和纵轴是基于共同高变基因的标准化 log1p CPM 表达值计算得到的前两个展示用 PCA 坐标，颜色表示该样本被归入的分子画像家族。

**小朋友版解释。** 这不是 UMAP 图，而是 PCA 图。可以把 PCA 想成把很多很多基因数字压到一张二维地图上：每个点是一份 RNA 样本，两个坐标表示它在这张地图上的位置，颜色表示模型觉得它属于哪一类分子画像。


## Supplementary Fig. 3a. RNA-text cosine controls

In [ ]:
# S3a. Distribution of RNA-text cosine values under paired and control settings.
data = read_table('s3_cosine_controls')
groups = [g for g in ['paired', 'shuffled', 'wrong_text'] if g in set(data['group'])]
if len(groups) < 2:
    groups = list(data['group'].drop_duplicates())[:4]
values = [data.loc[data['group'] == g, 'cosine'].dropna().to_numpy() for g in groups]

fig, ax = plt.subplots(figsize=(3.8, 2.9))
parts = ax.violinplot(values, showmeans=True, showmedians=False, showextrema=False)
for i, body in enumerate(parts['bodies']):
    body.set_facecolor([PALETTE['blue'], PALETTE['grey_light'], PALETTE['orange_light'], PALETTE['violet']][i % 4])
    body.set_alpha(0.65)
parts['cmeans'].set_color(PALETTE['black'])
ax.set_xticks(np.arange(1, len(groups) + 1), groups, rotation=20, ha='right')
ax.set_ylabel('RNA-text cosine')
ax.set_title('RNA-text cosine distributions')
clean_axis(ax)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s3a_paths = save_panel(fig, 'S3a_cosine_control_distributions')

**Caption (English).** Distribution of RNA-text cosine similarities for paired and control settings. The y axis shows cosine similarity; each violin summarizes one pairing condition.

**图注（中文）.** RNA 和文本相似度在真实配对和对照配对中的分布。纵坐标是 cosine 相似度，每个小提琴图表示一种配对方式。

**小朋友版解释.** 这张图看 RNA 和文字是不是配得上。真实配对如果更高，就像正确照片和说明更像一对。

## Main Fig. 2b. Paired RNA-text cosine

In [ ]:
# Figure 2b. Standard bar plot with error bars: paired text is closer.
data = read_table('figure_2_cosine_summary')
plot_data = data.set_index('group').loc[['shuffled', 'paired']].reset_index()

fig, ax = plt.subplots(figsize=(2.4, 2.6))
x = np.arange(len(plot_data))
y = plot_data['mean'].to_numpy()
yerr = np.vstack([y - plot_data['ci_low'].to_numpy(), plot_data['ci_high'].to_numpy() - y])

ax.bar(x, y, color=[PALETTE['grey_light'], PALETTE['blue']], width=0.62)
ax.errorbar(x, y, yerr=yerr, fmt='none', color=PALETTE['black'], capsize=3, lw=0.8)
ax.set_xticks(x, plot_data['group'])
ax.set_ylabel('RNA-text cosine')
ax.set_ylim(0, 0.78)
ax.set_title('Paired text is closer')
clean_axis(ax)
fig2b_paths = save_panel(fig, 'Figure_2b_bar_paired_text_cosine')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_2b_bar_paired_text_cosine -->
**Caption (English).** RNA profiles are closer to their paired descriptions than to shuffled descriptions. The y axis is RNA-text cosine similarity; higher bars mean that an RNA sample and a text description are nearer in the shared space.

**图注（中文）。** RNA 样本与其配对文字的距离比随机打乱文字更近。纵轴是 RNA-文字余弦相似度；柱子越高，说明 RNA 样本和文字描述在共享空间里越接近。

**小朋友版解释。** 灰色柱子是乱配的文字，蓝色柱子是正确配对的文字。蓝色更高，表示模型更容易把 RNA 找到它自己的说明书。


## Main Fig. 2c. Exact pairing lift

In [ ]:
# Figure 2c. Standard bar plot: exact RNA-to-text pairing lift.
data = read_table('figure_2_exact_retrieval')
plot_data = data[data['direction'] == 'rna_to_text'].copy()

fig, ax = plt.subplots(figsize=(2.7, 2.6))
x = np.arange(len(plot_data))
ax.bar(x, plot_data['lift_over_random'], color=PALETTE['blue_light'], width=0.65)
ax.set_xticks(x, plot_data['metric'])
ax.set_ylabel('lift over random')
ax.set_title('Exact pairing')
clean_axis(ax)
fig2c_paths = save_panel(fig, 'Figure_2c_bar_exact_pairing_lift')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_2c_bar_exact_pairing_lift -->
**Caption (English).** Exact RNA-to-text retrieval exceeds random expectation. The y axis reports lift over random matching; values above one indicate that the correct paired text is recovered more often than chance.

**图注（中文）。** RNA 到文字的精确配对高于随机水平。纵轴表示相对于随机匹配的提升倍数；数值大于 1 表明模型找到正确配对文字的频率超过碰运气。

**小朋友版解释。** 如果完全靠猜，分数大约是 1。柱子比 1 高，说明模型不是乱猜，而是真的学到了一些 RNA 和文字之间的对应关系。


## Supplementary Fig. 3b. Exact retrieval lift by direction

In [ ]:
# S3b. Exact retrieval lift by direction and rank.
data = read_table('s3_exact')
fig, ax = plt.subplots(figsize=(4.0, 2.8))
metrics = list(data['metric'].drop_duplicates())
directions = list(data['direction'].drop_duplicates())
x = np.arange(len(metrics))
width = 0.35 if len(directions) == 2 else 0.25
for i, direction in enumerate(directions):
    sub = data[data['direction'] == direction].set_index('metric').reindex(metrics).reset_index()
    offset = (i - (len(directions) - 1) / 2) * width
    ax.bar(x + offset, sub['lift_over_random'], width=width, label=direction, color=[PALETTE['blue'], PALETTE['orange'], PALETTE['green']][i % 3])
ax.set_xticks(x, metrics)
ax.set_ylabel('lift over random')
ax.set_title('Exact retrieval lift')
ax.legend(loc='best')
clean_axis(ax)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s3b_paths = save_panel(fig, 'S3b_exact_retrieval_lift_by_direction')

**Caption (English).** Exact retrieval lift over random expectation by retrieval direction and rank metric. The y axis reports lift over random, and bars separate RNA-to-text and text-to-RNA retrieval directions.

**图注（中文）.** 精确检索相对于随机水平的提升。纵坐标是比随机高多少，颜色区分 RNA 找文本和文本找 RNA 的方向。

**小朋友版解释.** 这张图像配对游戏：模型能不能从很多说明里找到正确的那一个。柱子越高，说明比瞎猜好很多。

## Main Fig. 2d. Broad semantic attribute recovery

In [ ]:
# Figure 2d. Standard grouped bar plot: broad semantic attributes.
data = read_table('figure_2_broad_retrieval')
plot_data = data[data['k'] == 1].set_index('label').loc[['site', 'tumor', 'disease']].reset_index()

fig, ax = plt.subplots(figsize=(4.2, 2.8))
x = np.arange(len(plot_data))
width = 0.35
ax.bar(x - width / 2, plot_data['random_baseline'], width=width, color=PALETTE['grey_light'], label='random')
ax.bar(x + width / 2, plot_data['value'], width=width, color=PALETTE['blue'], label='observed')
ax.errorbar(
    x + width / 2,
    plot_data['value'],
    yerr=np.vstack([plot_data['value'] - plot_data['ci_low'], plot_data['ci_high'] - plot_data['value']]),
    fmt='none', color=PALETTE['black'], capsize=3, lw=0.8,
)
ax.set_xticks(x, ['site', 'tumour status', 'disease family'])
ax.set_ylabel('top-1 semantic match')
ax.set_ylim(0, 1.03)
ax.set_title('Broad semantic attributes')
ax.legend(loc='best')
clean_axis(ax)
fig2d_paths = save_panel(fig, 'Figure_2d_grouped_bar_semantic_attributes')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_2d_grouped_bar_semantic_attributes -->
**Caption (English).** Broad sample attributes are recovered from RNA-text alignment. For tissue site, tumour status and disease family, blue bars show observed top-1 semantic matches and grey bars show random baselines.

**图注（中文）。** RNA-文字对齐可以恢复样本的宽泛属性。对于组织来源、肿瘤状态和疾病家族，蓝色柱表示观察到的 top-1 语义匹配，灰色柱表示随机基线。

**小朋友版解释。** 这里问模型三个大问题：来自哪里、是不是肿瘤、像哪类疾病。蓝色比灰色高，说明模型回答这些大方向时比随机猜更好。


## Supplementary Fig. 3c. Broad semantic retrieval rank curve

In [ ]:
# S3c. Broad semantic retrieval across rank cutoffs.
data = read_table('s3_broad')
fig, ax = plt.subplots(figsize=(4.0, 2.8))
for label, color in zip(['site', 'tumor', 'disease'], [PALETTE['blue'], PALETTE['green'], PALETTE['red']]):
    sub = data[data['label'] == label].sort_values('k')
    ax.plot(sub['k'], sub['value'], marker='o', color=color, label=label)
    if {'ci_low', 'ci_high'}.issubset(sub.columns):
        ax.fill_between(sub['k'].to_numpy(), sub['ci_low'].to_numpy(), sub['ci_high'].to_numpy(), color=color, alpha=0.12, lw=0)
ax.set_xlabel('rank cutoff k')
ax.set_ylabel('semantic match')
ax.set_ylim(0, 1.03)
ax.set_title('Broad semantic retrieval across k')
ax.legend(loc='best')
clean_axis(ax)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s3c_paths = save_panel(fig, 'S3c_broad_semantic_retrieval_rank_curve')

**Caption (English).** Broad semantic retrieval across rank cutoffs. The x axis shows the rank cutoff k; the y axis shows semantic match rate for site, tumour status and disease family.

**图注（中文）.** 宽泛语义检索随 rank cutoff 的变化。横坐标是允许看前 k 个结果，纵坐标是组织、肿瘤状态或疾病类别匹配率。

**小朋友版解释.** 这张图问：如果让模型给前几个答案，里面有没有答对大方向。线越高，说明越常答对。

# Evidence block 3. A single predefined disease name compresses molecular portraits

Claim tested here: a predefined disease-name call can be confident while the RNA portrait still shows several signals or weak support. Healthy and non-tumour metadata groups are handled separately rather than being forced into the frequent disease-name panels.

## Main Fig. 3a. Predefined disease-name assignment at the 0.70 operating point

In [ ]:
# Figure 3a. Standard grouped bar plot: predefined disease-label calls.
thresholds = read_table('figure_3_thresholds')
pool_summary = read_table('figure_3_external_pool_summary')
sub = thresholds[(thresholds['model'] == 'temperature_scaled_sgd') & (thresholds['confidence_threshold'] == 0.7)]
sub = sub.set_index('pool').loc[['External-180', 'MultiSource-450']].reset_index()
summ = pool_summary[pool_summary['model'] == 'temperature_scaled_sgd'].set_index('pool').loc[['External-180', 'MultiSource-450']].reset_index()

fig, ax = plt.subplots(figsize=(3.2, 2.7))
x = np.arange(len(sub))
width = 0.35
ax.bar(x - width / 2, sub['coverage'], width=width, color=PALETTE['blue_light'], label='label >=0.70')
ax.bar(x + width / 2, summ['highconf_overcall_70'], width=width, color=PALETTE['red'], label='uncertain but labelled')
ax.set_xticks(x, ['Ext-180', 'MS-450'])
ax.set_ylabel('fraction of profiles')
ax.set_ylim(0, max(0.22, float(max(sub['coverage'].max(), summ['highconf_overcall_70'].max())) * 1.35))
ax.set_title('Predefined disease labels')
ax.legend(loc='best')
clean_axis(ax)
fig3a_paths = save_panel(fig, 'Figure_3a_grouped_bar_predefined_label')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_3a_grouped_bar_predefined_label -->
**Caption (English).** Predefined disease-label calls in external profiles at a confidence threshold of 0.70. The x axis shows the two external sample pools. The y axis shows the fraction of profiles. The blue bar is the fraction receiving a predefined disease-family label with confidence >=0.70; the red bar is the fraction still given such a label although the portrait reader marked the sample as several-signal or weak-evidence. The 0.70 threshold is a moderate-high analysis threshold rather than a field standard; threshold sweeps are shown in Supplementary Fig. 4c.

**图注（中文）。** 外部样本中预设疾病标签的调用情况，置信度阈值为 0.70。横坐标是两个外部样本池，纵坐标是样本比例。蓝色柱表示有多少样本被预设疾病标签分类器以 >=0.70 的置信度给出疾病家族标签；红色柱表示这些样本虽然被分子画像读出为多个信号或证据较弱，但仍被给出单一疾病标签。0.70 是本文使用的中等偏高分析阈值，不是领域标准；阈值扫描见 Supplementary Fig. 4c。

**小朋友版解释。** 这张图是在问：如果模型只能从一张固定疾病名单里选答案，它会给多少样本一个很有把握的疾病名字？蓝色是“给了名字”，红色是“样本其实看起来不够单一或不够清楚，但还是被给了名字”。0.70 不是行业规定，只是我们选的一个比较严格但还不至于样本太少的检查线。


## Supplementary Fig. 4a. Internal reliability curve

In [ ]:
# S4a. Internal reliability curve.
data = read_table('s4_reliability')
fig, ax = plt.subplots(figsize=(3.4, 3.0))
for model, color in zip(data['model'].drop_duplicates(), [PALETTE['grey'], PALETTE['blue'], PALETTE['green']]):
    sub = data[(data['model'] == model) & data['accuracy'].notna()].sort_values('bin_mid')
    ax.plot(sub['mean_confidence'], sub['accuracy'], marker='o', color=color, label=model)
ax.plot([0, 1], [0, 1], ls='--', color=PALETTE['grey_light'], lw=1)
ax.set_xlabel('mean confidence')
ax.set_ylabel('accuracy')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_title('Internal reliability curve')
ax.legend(loc='best')
clean_axis(ax)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s4a_paths = save_panel(fig, 'S4a_internal_reliability_curve')

**Caption (English).** Internal reliability curve for predefined disease-label classifiers. The x axis shows mean confidence and the y axis shows empirical accuracy; the diagonal marks perfect calibration.

**图注（中文）.** 预设疾病标签分类器的内部可靠性曲线。横坐标是平均置信度，纵坐标是真实准确率，对角线表示理想校准。

**小朋友版解释.** 这张图看模型说“我很确定”时是不是真的靠谱。点越接近斜线，越诚实。

## Supplementary Fig. 4b. Internal risk-coverage curve

In [ ]:
# S4b. Internal risk-coverage curve.
data = read_table('s4_risk_coverage')
fig, ax = plt.subplots(figsize=(3.4, 3.0))
for model, color in zip(data['model'].drop_duplicates(), [PALETTE['grey'], PALETTE['blue'], PALETTE['green']]):
    sub = data[data['model'] == model].sort_values('coverage')
    ax.plot(sub['coverage'], sub['error_among_covered'], marker='o', color=color, label=model)
ax.set_xlabel('coverage')
ax.set_ylabel('error among covered')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, max(0.65, float(data['error_among_covered'].max()) * 1.08))
ax.set_title('Risk-coverage behaviour')
ax.legend(loc='best')
clean_axis(ax)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s4b_paths = save_panel(fig, 'S4b_internal_risk_coverage_curve')

**Caption (English).** Risk-coverage curve for predefined disease-label classifiers. The x axis shows the fraction of retained samples and the y axis shows error among retained predictions.

**图注（中文）.** 预设疾病标签分类器的风险-覆盖曲线。横坐标是保留多少样本，纵坐标是在这些保留样本中的错误率。

**小朋友版解释.** 这张图看如果只回答最有把握的问题，错误会不会减少。越往下越好。

## Supplementary Fig. 4c. Threshold sweep

In [ ]:
# S4c. Threshold sweep for predefined disease-label calls.
data = read_table('s4_external_thresholds')
plot_data = data[data['model'] == 'temperature_scaled_sgd'].copy()
fig, ax = plt.subplots(figsize=(4.0, 3.0))
for pool, color in zip(plot_data['pool'].drop_duplicates(), [PALETTE['blue'], PALETTE['orange']]):
    sub = plot_data[plot_data['pool'] == pool].sort_values('confidence_threshold')
    ax.plot(sub['confidence_threshold'], sub['coverage'], marker='o', color=color, label=f'{pool}: label retained')
    ax.plot(sub['confidence_threshold'], sub['overcall_among_covered_vs_openworld'], marker='s', ls='--', color=color, alpha=0.75, label=f'{pool}: uncertain but labelled')
ax.set_xlabel('confidence threshold')
ax.set_ylabel('fraction or rate')
ax.set_ylim(0, 1.02)
ax.set_title('Predefined-label threshold sweep')
ax.legend(loc='best', fontsize=8)
clean_axis(ax)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s4c_paths = save_panel(fig, 'S4c_external_threshold_sweep')

**Caption (English).** Confidence-threshold sweep for predefined disease-label calls in external profiles. The x axis shows the confidence threshold. Solid lines show the fraction of profiles retaining a predefined disease-family label above each threshold; dashed lines show the fraction of retained profiles that were still marked by the portrait reader as several-signal or weak-evidence. This panel shows why 0.70 was used as a moderate-high operating point in Fig. 3a, whereas 0.90 retains very few profiles. The 0.50, 0.70, 0.90 and 0.95 operating points are shown explicitly in Supplementary Fig. 4d.

**图注（中文）.** 外部样本中预设疾病标签调用的置信度阈值扫描。横坐标是置信度阈值。实线表示在每个阈值以上仍能保留预设疾病家族标签的样本比例；虚线表示这些被保留样本中，仍被分子画像读为多个信号或证据较弱的比例。该图说明为什么 Figure 3a 使用 0.70 作为中等偏高的工作阈值，而 0.90 会只保留很少样本。0.50、0.70、0.90 和 0.95 这几个具体操作点在 Supplementary Fig. 4d 中单独展示。

**小朋友版解释.** 这张图是在移动“多有把握才算数”的线。线往右走，要求越来越严格，能留下的样本会越来越少。0.70 像一条比较严格但还能留下足够样本的线；0.90 已经很严格，0.95 会在下一张图里单独展示。


## Supplementary Fig. 4d. Explicit 0.50, 0.70, 0.90 and 0.95 operating points

In [ ]:
# S4d. Explicit operating points for 0.50, 0.70, 0.90 and 0.95.
pred = read_table('s4_external_predictions')
model = 'temperature_scaled_sgd'
confidence_col = 'closed_set_calibrated_confidence'
uncertain_col = 'closed_set_over_openworld_mixed_or_unsupported'
thresholds = [0.50, 0.70, 0.90, 0.95]
rows = []
for pool, group in pred[pred['model'] == model].groupby('pool'):
    total = len(group)
    for threshold in thresholds:
        covered = group[group[confidence_col] >= threshold]
        n_covered = len(covered)
        rows.append({
            'pool': pool,
            'threshold': threshold,
            'n_covered': n_covered,
            'coverage': n_covered / total if total else 0.0,
            'uncertain_among_covered': covered[uncertain_col].astype(bool).mean() if n_covered else np.nan,
        })
operating = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.9), sharex=True)
pools = ['External-180', 'MultiSource-450']
x = np.arange(len(thresholds))
width = 0.36
colors = {'External-180': PALETTE['blue'], 'MultiSource-450': PALETTE['orange']}

for i, pool in enumerate(pools):
    sub = operating[operating['pool'] == pool].set_index('threshold').reindex(thresholds).reset_index()
    offset = (i - 0.5) * width
    bars = axes[0].bar(x + offset, sub['coverage'], width=width, color=colors[pool], label=pool)
    for bar, n in zip(bars, sub['n_covered']):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015, f'n={int(n)}', ha='center', va='bottom', fontsize=7, rotation=90)
    axes[1].bar(x + offset, sub['uncertain_among_covered'], width=width, color=colors[pool], label=pool)

axes[0].set_title('Profiles retained')
axes[0].set_ylabel('fraction of profiles')
axes[0].set_ylim(0, 0.52)
axes[1].set_title('Uncertain but labelled')
axes[1].set_ylabel('fraction among retained')
axes[1].set_ylim(0, 1.05)
for ax in axes:
    ax.set_xticks(x, [f'{t:.2f}' for t in thresholds])
    ax.set_xlabel('confidence threshold')
    clean_axis(ax)
axes[1].legend(loc='lower left', bbox_to_anchor=(1.02, 0.02))
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s4d_paths = save_panel(fig, 'S4d_threshold_operating_points')

**Caption (English).** Explicit operating-point comparison for predefined disease-label calls. The left panel shows how many profiles remain when the confidence threshold is set to 0.50, 0.70, 0.90 or 0.95; labels above bars show the retained sample count. The right panel shows, among retained profiles, the fraction still marked by the portrait reader as several-signal or weak-evidence. The 0.50 threshold retains more profiles but still contains many uncertain-but-labelled cases, whereas 0.90 and 0.95 retain too few profiles for a reliable main-panel comparison.

**图注（中文）.** 预设疾病标签调用在几个具体置信度阈值下的对照。左图表示阈值设为 0.50、0.70、0.90 或 0.95 时还剩多少样本，柱子上方的 n 是保留下来的样本数。右图表示这些被保留样本中，有多少仍被分子画像读为多个信号或证据较弱。0.50 会留下更多样本，但仍有很多“不够清楚却被贴标签”的情况；0.90 和 0.95 又太严格，留下的样本太少，不适合作为主图比较。

**小朋友版解释.** 这张图把几条“及格线”摆在一起看。0.50 像比较宽松的及格线，留下的人多；0.70 是我们主图用的线，比较严格但还剩一些样本；0.90 和 0.95 太严格，很多样本都被踢掉了，所以不能只看它们来讲主要故事。


## Main Fig. 3b. Portrait status fractions

In [ ]:
# Figure 3b. Standard stacked bar plot: portrait status fractions.
data = read_table('figure_3_sample_transitions')
data = data.copy()
data['reader_status'] = data['resolved_status'].map(STATUS_KEY_MAP).fillna(data['resolved_status'])
status = data.groupby(['pool', 'reader_status']).size().reset_index(name='n')
pools = ['External-180', 'MultiSource-450']
status_order = ['single_clear_signal', 'mixed', 'unsupported']
status_colors = {'single_clear_signal': PALETTE['green'], 'mixed': PALETTE['orange_light'], 'unsupported': PALETTE['grey']}

fig, ax = plt.subplots(figsize=(3.8, 2.7))
bottom = np.zeros(len(pools))
for key in status_order:
    vals = []
    for pool in pools:
        n = status.loc[(status['pool'] == pool) & (status['reader_status'] == key), 'n'].sum()
        total = status.loc[status['pool'] == pool, 'n'].sum()
        vals.append(n / total if total else 0)
    ax.bar(pools, vals, bottom=bottom, color=status_colors[key], label=STATUS_LABELS[key])
    bottom += np.array(vals)

ax.set_ylabel('fraction of profiles')
ax.set_ylim(0, 1.02)
ax.set_title('Portrait status')
ax.legend(loc='best')
clean_axis(ax)
fig3b_paths = save_panel(fig, 'Figure_3b_stacked_bar_portrait_status')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_3b_stacked_bar_portrait_status -->
**Caption (English).** Molecular portrait status across external pools. Each stacked bar sums to one; colours show the fraction of profiles with one clear signal, several signals, or weak evidence.

**图注（中文）。** 外部样本池中的分子画像状态。每根堆叠柱总和为 1；不同颜色表示单一清晰信号、多个信号或证据较弱样本所占比例。

**小朋友版解释。** 每根柱子都是一篮子样本。绿色像“信号很清楚”，橙色像“有好几个信号混在一起”，灰色像“证据还不够强”。


## Supplementary Fig. 5a. Raw disease family to portrait status

In [ ]:
# S5a. Raw disease family to resolved status.
data = read_table('s5_status_transition')
data = data.copy()
data['reader_status'] = data['resolved_status'].map(STATUS_KEY_MAP).fillna(data['resolved_status'])
plot_data = data.groupby(['raw_disease_family', 'reader_status'], as_index=False)['n'].sum()
matrix = row_normalized_pivot(plot_data, 'raw_disease_family', 'reader_status', top_n=10)

fig, ax = plt.subplots(figsize=(4.2, 3.4))
im = ax.imshow(matrix.to_numpy(), aspect='auto', cmap=LinearSegmentedColormap.from_list('white_blue', ['#FFFFFF', PALETTE['blue_soft'], PALETTE['blue']]), vmin=0, vmax=1)
status_display = STATUS_LABELS
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(status_display.get(x, x), 14) for x in matrix.columns], rotation=35, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [wrap_label(x.replace('_', ' '), 22) for x in matrix.index])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, label='row fraction')
ax.set_title('Raw disease family to portrait status')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s5a_paths = save_panel(fig, 'S5a_transition_raw_disease_to_status')

**Caption (English).** Transition from raw disease family to portrait status. Rows are raw disease families, columns are resolved portrait statuses, and colours show row-normalized fractions.

**图注（中文）.** 从原始疾病类别到画像状态的转换。行是原始疾病类别，列是画像状态，颜色表示每一行内部的比例。

**小朋友版解释.** 这张图看一个旧疾病名字最后会被读成哪种画像状态。颜色越深，说明越常走到那个结果。

## Supplementary Fig. 5b. Raw disease family to resolved family

In [ ]:
# S5b. Raw disease family to resolved disease-family reading.
data = read_table('s5_family_transition')
plot_data = data.groupby(['raw_disease_family', 'resolved_disease_family'], as_index=False)['n'].sum()
matrix = row_normalized_pivot(plot_data, 'raw_disease_family', 'resolved_disease_family', top_n=10)
# Keep the most frequent resolved columns to avoid an unreadable matrix.
keep_cols = matrix.sum(axis=0).sort_values(ascending=False).head(10).index
matrix = matrix[keep_cols]

fig, ax = plt.subplots(figsize=(5.2, 3.6))
im = ax.imshow(matrix.to_numpy(), aspect='auto', cmap=LinearSegmentedColormap.from_list('white_blue', ['#FFFFFF', PALETTE['blue_soft'], PALETTE['blue']]), vmin=0, vmax=float(matrix.max().max()))
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(x.replace('_', ' '), 14) for x in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [wrap_label(x.replace('_', ' '), 22) for x in matrix.index])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, label='row fraction')
ax.set_title('Raw disease family to resolved family reading')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s5b_paths = save_panel(fig, 'S5b_transition_raw_to_resolved_family')

**Caption (English).** Transition from raw disease family to resolved disease-family reading. Rows are raw disease families, columns are resolved readings, and colours show row-normalized fractions.

**图注（中文）.** 从原始疾病类别到解析后的疾病家族读法。行是原始疾病类别，列是新的解析读法，颜色表示每一行内部的比例。

**小朋友版解释.** 这张图像翻译表：旧名字经过画像系统后，会变成哪些更细的解释。颜色越深表示越常见。

## Main Fig. 3c. Portrait diversity within disease names

In [ ]:
# Figure 3c. Standard horizontal bar plot: portrait diversity within disease names.
data = read_table('figure_3_predefined_label_diversity')
plot_data = data[data['n_samples'] >= 10].sort_values('n_samples', ascending=False).head(8)
plot_data = plot_data.sort_values('portrait_diversity')

fig, ax = plt.subplots(figsize=(3.45, 3.0))
y = np.arange(len(plot_data))
ax.barh(y, plot_data['portrait_diversity'], color=PALETTE['violet'])
ax.set_yticks(y, [wrap_label(label, 16) for label in plot_data['display_label']])
for yi, row in enumerate(plot_data.itertuples(index=False)):
    ax.text(row.portrait_diversity + 0.025, yi, f'n={int(row.n_samples)}', va='center', ha='left', fontsize=8)
ax.set_xlim(0, 1.08)
ax.set_xlabel('portrait diversity')
ax.set_title('Diversity within disease names')
clean_axis(ax, grid='x')
fig3c_paths = save_panel(fig, 'Figure_3c_horizontal_bar_disease_name_diversity')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_3c_horizontal_bar_disease_name_diversity -->
**Caption (English).** Samples sharing the same predefined disease name can contain multiple portrait patterns. The y axis lists frequent disease labels, the x axis measures portrait diversity within each disease label, and n gives the number of profiles carrying that label.

**图注（中文）。** 拥有同一个预设疾病名称的样本内部可能包含多种分子画像。纵轴列出常见疾病标签，横轴表示每个疾病标签内部的画像多样性，n 表示该疾病标签下的样本数。

**小朋友版解释。** 这里每一行都是一个疾病名字。横条越长，说明同一个疾病名字下面装着越多不同颜色的积木，也就是样本的分子画像不完全一样。


## Main Fig. 3d. Predefined disease names crossed with portrait families

In [ ]:
# Figure 3d. Standard heatmap: predefined disease names crossed with portrait groups.
data = read_table('figure_3_predefined_label_portraits')
label_col = 'closed_set_disease_family'
plot_data = data.copy()
plot_data['total'] = plot_data.drop(columns=[label_col]).sum(axis=1)
plot_data = plot_data.sort_values('total', ascending=False).head(8)
matrix = plot_data.set_index(label_col)[[c for c in STATE_ORDER if c in plot_data.columns]]
matrix = matrix.div(matrix.sum(axis=1), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(4.8, 3.4))
cmap = LinearSegmentedColormap.from_list('white_blue', ['#FFFFFF', '#D7E3F1', PALETTE['blue']])
im = ax.imshow(matrix.to_numpy(), aspect='auto', cmap=cmap, vmin=0, vmax=float(matrix.max().max()))
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(STATE_LABELS.get(c, c), 10) for c in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [wrap_label(x.replace('_', ' '), 18) for x in matrix.index])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(im, ax=ax, label='fraction within disease name')
ax.set_xlabel('molecular portrait family')
ax.set_ylabel('predefined disease name')
ax.set_title('One disease name, several portraits')
fig3d_paths = save_panel(fig, 'Figure_3d_heatmap_disease_by_portrait')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_3d_heatmap_disease_by_portrait -->
**Caption (English).** Predefined disease names cross multiple portrait families. Rows are predefined disease names, columns are molecular portrait families, and colour shows the fraction within each disease name assigned to each portrait family. Each row is normalized separately, so the colour is not the fraction of all samples.

**图注（中文）。** 预设疾病名称会横跨多个分子画像家族。行是预设疾病名，列是分子画像家族，颜色表示在每一个疾病名内部，有多少比例落在对应画像家族。每一行单独归一化，所以颜色不是全体样本中的比例。

**小朋友版解释。** 左边每一行是一个疾病名字，上面每一列是一种分子画像。我们只在同一行里面比较颜色：颜色越深，说明这个疾病名字里的样本越多长得像那种画像。不能拿不同行的颜色当作全体样本数量来比。


## Supplementary Fig. 5c. Healthy, non-tumour and tumour metadata groups

In [ ]:
# S5c. Expected healthy/non-tumour/tumour metadata groups to molecular portrait families.
data = read_table('s5_expected_transition').copy()

def expected_metadata_group(label):
    label = str(label)
    if label == 'healthy_control':
        return 'healthy metadata'
    if 'non_tumor' in label or label in {'autoimmune_inflammatory', 'liver_metabolic'}:
        return 'non-tumour metadata'
    return 'tumour metadata'

group_order = ['healthy metadata', 'non-tumour metadata', 'tumour metadata']
data['expected_metadata_group'] = data['expected_disease_family'].map(expected_metadata_group)

counts = (
    data.groupby(['expected_metadata_group', 'semantic_state_family'])
    .size()
    .reset_index(name='n')
)
count_matrix = (
    counts.pivot(index='expected_metadata_group', columns='semantic_state_family', values='n')
    .reindex(group_order)
    .reindex(columns=STATE_ORDER, fill_value=0)
    .fillna(0)
    .astype(int)
)
fraction_matrix = count_matrix.div(count_matrix.sum(axis=1), axis=0).fillna(0)

source_data = (
    count_matrix.stack()
    .rename('n')
    .reset_index()
    .rename(columns={'level_1': 'semantic_state_family'})
)
source_data['figure_label'] = source_data['semantic_state_family'].map(STATE_LABELS)
source_data['group_total'] = source_data['expected_metadata_group'].map(count_matrix.sum(axis=1).to_dict())
source_data['fraction_within_expected_group'] = source_data['n'] / source_data['group_total']
source_out = WORK_DIR / 'source_data_extra' / 'supplement_s5c_expected_metadata_group_to_portrait.csv'
source_out.parent.mkdir(parents=True, exist_ok=True)
source_data.to_csv(source_out, index=False)
print('source data:', source_out)
display(source_data[['expected_metadata_group', 'figure_label', 'n', 'group_total', 'fraction_within_expected_group']].head())

fig, ax = plt.subplots(figsize=(5.7, 2.9))
y = np.arange(len(fraction_matrix))
left = np.zeros(len(fraction_matrix))
for state in STATE_ORDER:
    vals = fraction_matrix[state].to_numpy(dtype=float)
    if vals.sum() == 0:
        continue
    ax.barh(
        y,
        vals,
        left=left,
        color=STATE_COLORS.get(state, PALETTE['grey']),
        label=STATE_LABELS.get(state, state),
        height=0.62,
        edgecolor='white',
        linewidth=0.5,
    )
    left += vals

row_totals = count_matrix.sum(axis=1)
y_labels = [f"{label}\n(n={int(row_totals.loc[label])})" for label in fraction_matrix.index]
ax.set_yticks(y, y_labels)
ax.invert_yaxis()
ax.set_xlim(0, 1.0)
ax.set_xlabel('fraction within expected metadata group')
ax.set_title('Expected metadata groups to portrait families')
ax.legend(loc='upper center', bbox_to_anchor=(0.50, -0.26), ncol=4, handlelength=1.0, columnspacing=0.8)
clean_axis(ax, grid='x')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s5c_paths = save_panel(fig, 'S5c_expected_metadata_group_to_portrait')

**Caption (English).** Expected disease-family metadata groups crossed with molecular portrait families. Rows group external profiles by the original expected metadata, not by the calibrated predefined disease-label classifier: healthy control, non-tumour metadata contexts and tumour metadata contexts. Bar segments show the fraction of profiles within each metadata group assigned to each molecular portrait family; n gives the number of profiles in the row. This panel explains why Fig. 3c,d focus on frequent predefined disease labels while healthy and non-tumour samples are analysed separately from expected metadata.

**图注（中文）.** expected disease-family metadata 分组与分子画像家族的对应关系。每一行按照外部样本原始 expected metadata 分组，而不是按照校准后的 predefined disease-label classifier 分组：healthy control、non-tumour metadata contexts 和 tumour metadata contexts。每个颜色段表示该 metadata 组内有多少比例的样本被分到对应分子画像家族；n 是该行样本数。该图解释了为什么 Fig. 3c,d 只展示高频 predefined disease labels，而 healthy 和 non-tumour 样本需要用 expected metadata 单独分析。

**小朋友版解释。** 这张图不是问“模型硬给它叫什么病名”，而是先看数据原本写着它是健康、非肿瘤还是肿瘤。然后看这些样本分别被 RNA 画像读成了什么颜色。这样就能看到：健康和非肿瘤样本没有被丢掉，只是它们不适合挤进 Fig. 3c,d 那张“最常见疾病名字”的主图里。

# Evidence block 4. Disease-associated portrait composition

Claim tested here: disease labels are associated with characteristic mixtures of molecular portrait families. This supports a bounded disease-information statement, not a diagnostic claim.

## Supplementary Fig. 11a. Disease labels mapped to portrait compositions

In [ ]:
# S11a. Disease labels mapped to molecular portrait compositions.
data = read_table('s11_composition_long').copy()
summary = read_table('T13_portrait_composition_disease_information/t13_disease_portrait_composition_summary') if False else None
order_rows = data.groupby('disease_label')['disease_n'].max().sort_values(ascending=False).index.tolist()
cols = [c for c in STATE_ORDER if c in set(data['portrait_family'])]
heat = data.pivot_table(index='disease_label', columns='portrait_family', values='fraction', aggfunc='sum', fill_value=0)
heat = heat.reindex(index=order_rows, columns=cols).fillna(0)
fig, ax = plt.subplots(figsize=(5.8, 4.8))
im = ax.imshow(heat.values, aspect='auto', cmap='Blues', vmin=0, vmax=max(0.5, float(heat.values.max())))
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(portrait_display_labels(cols, width=16), rotation=45, ha='right')
ax.set_yticks(range(len(heat.index)))
row_labels = [wrap_label(str(x).replace('_', ' '), 24) for x in heat.index]
ax.set_yticklabels(row_labels)
ax.set_xlabel('Molecular portrait family')
ax.set_ylabel('Expected disease-family label')
ax.set_title('Disease-specific portrait compositions')
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label('Fraction within disease label')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        if val >= 0.18:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6.5, color='white' if val > 0.32 else PALETTE['black'])
apply_font_sizes(fig)
s11a_paths = save_panel(fig, 'S11a_disease_portrait_composition_heatmap')

**Caption (English).** Disease labels mapped to molecular portrait compositions. Rows are expected disease-family labels with at least ten samples, columns are molecular portrait families, and colour shows the fraction of samples within each disease label.

**图注（中文）.** 疾病标签与分子画像组成的对应关系。每一行是至少包含 10 个样本的 expected disease-family 标签，每一列是一个分子画像家族，颜色表示该疾病标签内部有多少比例的样本落入对应画像。

**小朋友版解释.** 每一行像一个疾病班级，每一列像一种画像颜色。颜色越深，说明这个疾病班级里越多样本长得像这种画像。这个图不是说一个疾病只等于一种画像，而是看它由哪些画像比例组成。

## Supplementary Fig. 11b. Disease prediction from portrait and molecular features

In [ ]:
# S11b. Disease prediction from portrait and molecular features.
data = read_table('s11_model_performance').copy()
preferred = [
    'metadata_controls',
    'portrait_family',
    'controls_plus_portrait',
    'marker_pathway',
    'controls_plus_marker_pathway',
    'controls_plus_portrait_marker_pathway',
]
data = data[data['model'].isin(preferred)].copy()
data['model'] = pd.Categorical(data['model'], categories=preferred, ordered=True)
data = data.sort_values('model')
fig, ax = plt.subplots(figsize=(5.2, 3.2))
colors = [PALETTE['grey'], PALETTE['blue'], PALETTE['blue_light'], PALETTE['green'], PALETTE['green_light'], PALETTE['violet']]
ax.barh(range(len(data)), data['balanced_accuracy_mean'], xerr=data['balanced_accuracy_sem'], color=colors[:len(data)], edgecolor='none')
ax.set_yticks(range(len(data)))
ax.set_yticklabels([wrap_label(x, 30) for x in data['model_label']])
ax.set_xlabel('Balanced accuracy')
ax.set_title('Disease information in portrait composition')
ax.set_xlim(0, max(0.85, float(data['balanced_accuracy_mean'].max() + data['balanced_accuracy_sem'].max() + 0.05)))
clean_axis(ax, grid='x')
apply_font_sizes(fig)
s11b_paths = save_panel(fig, 'S11b_disease_prediction_model_comparison')

**Caption (English).** Cross-validated disease prediction from portrait and molecular features. Bars show balanced accuracy across 14 disease labels; error bars show the standard error across folds.

**图注（中文）.** 使用画像和分子特征预测疾病标签的交叉验证结果。柱子表示 14 个疾病标签上的 balanced accuracy，误差线表示不同 fold 之间的标准误。

**小朋友版解释.** 这里像是在考试：只给模型看不同信息，问它能不能猜出疾病标签。画像信息能帮忙，但不能把它当成临床诊断。

## Supplementary Fig. 11c. Portrait increment after tissue/source controls

In [ ]:
# S11c. Portrait information after tissue and source controls.
data = read_table('s11_permutation').copy()
real = data[data['test'] == 'real_increment_over_controls']['delta_balanced_accuracy'].iloc[0]
perm = data[data['test'] == 'within_tissue_source_permutation']['delta_balanced_accuracy']
p_value = data['one_sided_p_value_for_real_delta'].dropna().iloc[0]
fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.hist(perm, bins=18, color=PALETTE['grey_light'], edgecolor='white', label='shuffled within tissue/source')
ax.axvline(real, color=PALETTE['red'], lw=2.0, label=f'real increment = {real:.3f}')
ax.axvline(0, color=PALETTE['black'], lw=0.8, ls=':')
ax.set_xlabel('Increment in balanced accuracy')
ax.set_ylabel('Permutation count')
ax.set_title('Portrait increment after controls')
ax.text(0.98, 0.92, f'one-sided P = {p_value:.3f}', transform=ax.transAxes, ha='right', va='top', fontsize=8)
ax.legend(loc='upper left')
clean_axis(ax, grid='y')
apply_font_sizes(fig)
s11c_paths = save_panel(fig, 'S11c_portrait_increment_after_controls_permutation')

**Caption (English).** Portrait-family information after tissue and source controls. The red line shows the real gain in balanced accuracy after adding portrait family to metadata controls; grey bars show gains after portrait labels were shuffled within tissue/source strata.

**图注（中文）.** 控制 tissue 和 source 后，分子画像家族还能提供多少额外疾病相关信息。红线是真实加入画像后的 balanced accuracy 增益；灰色柱子是在 tissue/source 分层内打乱画像标签后的增益分布。

**小朋友版解释.** 这张图是在问：画像是不是只是在偷偷记住样本来自哪里？我们把同一组织和来源里的画像打乱再比较。如果真实红线比大多数灰柱子更靠右，说明画像里还有不是简单来源标签能解释的信息。

## Supplementary Fig. 11d. Disease-level composition identifiability

In [ ]:
# S11d. Disease-level portrait-composition identifiability.
data = read_table('s11_group_identification').copy().sort_values('composition_id_accuracy')
fig, ax = plt.subplots(figsize=(4.8, 3.8))
ax.barh(range(len(data)), data['composition_id_accuracy'], color=PALETTE['blue'], edgecolor='none')
ax.set_yticks(range(len(data)))
ax.set_yticklabels([wrap_label(str(x).replace('_', ' '), 24) for x in data['disease_label']])
ax.set_xlabel('Composition-identification accuracy')
ax.set_title('Disease-level composition identifiability')
ax.set_xlim(0, 1.0)
clean_axis(ax, grid='x')
apply_font_sizes(fig)
s11d_paths = save_panel(fig, 'S11d_group_level_composition_identification')

**Caption (English).** Disease-level portrait-composition identifiability. Repeated train-test splits estimated whether each disease label could be identified from its held-out portrait composition using nearest-composition matching.

**图注（中文）.** 疾病层面的画像组成可识别性。通过重复 train-test split，估计只看 held-out 的画像组成，能否用最近组成匹配识别出对应疾病标签。

**小朋友版解释.** 这像把每种疾病的“画像配方”遮住名字后拿去匹配。如果某个疾病的配方比较稳定，就更容易被找回。

# Evidence block 5. Portrait families have independent biological grounding

Claim tested here: portrait families correspond to marker programmes, pathway programmes and cell-composition estimates, and textual portrait claims can be checked against independent molecular evidence.

## Main Fig. 4a. Marker-programme grounding

In [ ]:
# Figure 4a. Standard heatmap: marker programmes.
data = read_table('figure_4_marker_heatmap_matrix')
matrix = data.set_index('semantic_state_family')

fig, ax = plt.subplots(figsize=(4.0, 2.8))
cmap = LinearSegmentedColormap.from_list('blue_white_red', ['#2F5D8C', '#F8F8F8', PALETTE['red']])
im = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', cmap=cmap, vmin=-1.1, vmax=1.1)
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(c, 12) for c in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [STATE_LABELS.get(i, i) for i in matrix.index])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(im, ax=ax, label='z score')
ax.set_title('Marker programmes')
fig4a_paths = save_panel(fig, 'Figure_4a_heatmap_marker_programmes')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_4a_heatmap_marker_programmes -->
**Caption (English).** Marker-gene programmes vary across portrait families. Rows are molecular portrait families, columns are marker programmes, and colour indicates standardized enrichment or depletion.

**图注（中文）。** 标志基因程序在不同分子画像家族中呈现差异。行是画像家族，列是标志基因程序，颜色表示标准化后的富集或降低。

**小朋友版解释。** 这张热图像一张课程表：行是不同画像，列是不同基因信号。红色更强，蓝色更弱，说明不同画像有不同的基因特征。


## Supplementary Fig. 6a. Full marker-programme heatmap

In [ ]:
# S6a. Full marker-programme heatmap.
data = read_table('s6_marker')
value_cols = [c for c in data.columns if c.endswith('_z')]
weighted = []
for state, g in data.groupby('semantic_state_family'):
    row = {'semantic_state_family': state}
    w = g['n'].to_numpy(dtype=float)
    for col in value_cols:
        row[col] = np.average(g[col].to_numpy(dtype=float), weights=w)
    weighted.append(row)
matrix = pd.DataFrame(weighted).set_index('semantic_state_family').reindex([s for s in STATE_ORDER if s in data['semantic_state_family'].unique()])
matrix.columns = [c.replace('_z', '').replace('_', ' ') for c in matrix.columns]

fig, ax = plt.subplots(figsize=(5.4, 3.2))
cmap = LinearSegmentedColormap.from_list('blue_white_red', ['#2F5D8C', '#F8F8F8', PALETTE['red']])
im = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', cmap=cmap, vmin=-1.1, vmax=1.1)
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(c, 12) for c in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [STATE_LABELS.get(i, i) for i in matrix.index])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, label='weighted mean z score')
ax.set_title('Full marker-programme grounding')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s6a_paths = save_panel(fig, 'S6a_full_marker_programme_heatmap')

**Caption (English).** Full marker-programme grounding by portrait group. Rows are portrait groups, columns are marker programmes, and colours show weighted mean z scores.

**图注（中文）.** 不同画像组的完整 marker programme 支撑。行是画像组，列是 marker programme，颜色表示加权平均 z score。

**小朋友版解释.** 这张图看每类画像有哪些生物线索更强。红色更强，蓝色更弱。

## Main Fig. 4b. Pathway-programme grounding

In [ ]:
# Figure 4b. Standard heatmap: pathway programmes.
data = read_table('figure_4_pathway_heatmap_matrix')
matrix = data.set_index('semantic_state_family')

fig, ax = plt.subplots(figsize=(4.0, 2.8))
cmap = LinearSegmentedColormap.from_list('blue_white_red', ['#2F5D8C', '#F8F8F8', PALETTE['red']])
im = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', cmap=cmap, vmin=-1.1, vmax=1.1)
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(c, 12) for c in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [STATE_LABELS.get(i, i) for i in matrix.index])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(im, ax=ax, label='score')
ax.set_title('Pathway programmes')
fig4b_paths = save_panel(fig, 'Figure_4b_heatmap_pathway_programmes')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_4b_heatmap_pathway_programmes -->
**Caption (English).** Pathway programmes provide pathway-level support for portrait families. Rows are molecular portrait families, columns are biological pathways, and colour reports programme scores.

**图注（中文）。** 通路程序从生物通路层面支持分子画像家族。行是画像家族，列是生物通路，颜色表示通路程序得分。

**小朋友版解释。** 如果 Figure 4a 看的是单个基因小队，这里看的就是更大的生物过程。颜色告诉我们每种画像更像哪些生物过程。


## Supplementary Fig. 6b. Full pathway-programme heatmap

In [ ]:
# S6b. Full pathway-programme heatmap.
data = read_table('s6_pathway')
value_cols = [c for c in data.columns if c.endswith('_score')]
weighted = []
for state, g in data.groupby('semantic_state_family'):
    row = {'semantic_state_family': state}
    w = g['n'].to_numpy(dtype=float)
    for col in value_cols:
        row[col] = np.average(g[col].to_numpy(dtype=float), weights=w)
    weighted.append(row)
matrix = pd.DataFrame(weighted).set_index('semantic_state_family').reindex([s for s in STATE_ORDER if s in data['semantic_state_family'].unique()])
matrix.columns = [c.replace('_score', '').replace('_', ' ') for c in matrix.columns]

fig, ax = plt.subplots(figsize=(6.0, 3.2))
cmap = LinearSegmentedColormap.from_list('blue_white_red', ['#2F5D8C', '#F8F8F8', PALETTE['red']])
im = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', cmap=cmap, vmin=-1.1, vmax=1.1)
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(c, 12) for c in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [STATE_LABELS.get(i, i) for i in matrix.index])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, label='weighted mean score')
ax.set_title('Full pathway-programme grounding')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s6b_paths = save_panel(fig, 'S6b_full_pathway_programme_heatmap')

**Caption (English).** Full pathway-programme grounding by portrait group. Rows are portrait groups, columns are pathway programmes, and colours show weighted mean scores.

**图注（中文）.** 不同画像组的完整通路支撑。行是画像组，列是通路 programme，颜色表示加权平均分数。

**小朋友版解释.** 这张图看每类画像像不像某些生物过程。红色说明这个过程更明显。

## Main Fig. 4c. EPIC and MCP-counter checks

In [ ]:
# Figure 4c. Standard scatter/dot plot: deconvolution checks.
epic = read_table('figure_4_epic_effects')
mcp = read_table('figure_4_mcp_effects')
combined = pd.concat([epic.assign(source='EPIC'), mcp.assign(source='MCP-counter')], ignore_index=True)
selected_scores = [
    'epic_immune_fraction_z',
    'epic_caf_fraction_z',
    'epic_endothelial_fraction_z',
    'mcp_t_cells_z',
    'mcp_cytotoxic_lymphocytes_z',
    'mcp_nk_cells_z',
    'mcp_endothelial_cells_z',
    'mcp_fibroblasts_z',
]
plot_data = combined[
    combined['semantic_state_family'].isin(STATE_ORDER[:5]) & combined['score'].isin(selected_scores)
].copy()
score_labels = {
    'epic_immune_fraction_z': 'EPIC\nImm.',
    'epic_caf_fraction_z': 'EPIC\nCAF',
    'epic_endothelial_fraction_z': 'EPIC\nEndo.',
    'mcp_t_cells_z': 'MCP\nT',
    'mcp_cytotoxic_lymphocytes_z': 'MCP\nCyto.',
    'mcp_nk_cells_z': 'MCP\nNK',
    'mcp_endothelial_cells_z': 'MCP\nEndo.',
    'mcp_fibroblasts_z': 'MCP\nFib.',
}
plot_data['score_label'] = plot_data['score'].map(score_labels)
plot_data['state_label'] = plot_data['semantic_state_family'].map(STATE_LABELS)
plot_data['dot_size'] = 12 + 10 * np.clip(-np.log10(plot_data['mannwhitney_p'].clip(lower=1e-12)), 0, 8)
x_labels = list(dict.fromkeys(plot_data['score_label'].tolist()))
y_labels = [STATE_LABELS[s] for s in STATE_ORDER[:5]]

fig, ax = plt.subplots(figsize=(5.0, 3.0))
scatter = ax.scatter(
    [x_labels.index(x) for x in plot_data['score_label']],
    [y_labels.index(y) for y in plot_data['state_label']],
    s=plot_data['dot_size'],
    c=plot_data['cohen_d'],
    cmap=LinearSegmentedColormap.from_list('effect', ['#2F5D8C', '#F8F8F8', PALETTE['red']]),
    vmin=-1.2,
    vmax=1.2,
    edgecolor='#333333',
    linewidth=0.25,
)
ax.set_xticks(np.arange(len(x_labels)), x_labels, rotation=0, ha='center')
ax.set_yticks(np.arange(len(y_labels)), y_labels)
ax.set_xlim(-0.6, len(x_labels) - 0.4)
ax.set_ylim(len(y_labels) - 0.5, -0.5)
fig.colorbar(scatter, ax=ax, label='effect size')
ax.set_title('Deconvolution checks')
clean_axis(ax, grid=None)
fig4c_paths = save_panel(fig, 'Figure_4c_scatter_deconvolution_checks')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_4c_scatter_deconvolution_checks -->
**Caption (English).** Cell-composition estimates are compared with portrait families. The x axis lists deconvolution-derived cell or stromal scores, the y axis lists portrait families, dot colour shows effect size, and dot size reflects statistical support.

**图注（中文）。** 将细胞组成估计与分子画像家族进行比较。横轴是反卷积得到的细胞或基质相关得分，纵轴是画像家族，点的颜色表示效应大小，点的大小表示统计支持强弱。

**小朋友版解释。** 每个圆点是在问：这种画像是不是和某类细胞信号有关？颜色越偏红代表更强，点越大代表证据更稳。


## Supplementary Fig. 6c. Full deconvolution effect heatmap

In [ ]:
# S6c. Deconvolution effect-size heatmap across selected cell-state axes.
epic = read_table('s6_epic')
mcp = read_table('s6_mcp')
data = pd.concat([epic.assign(source='EPIC'), mcp.assign(source='MCP-counter')], ignore_index=True)
selected = [
    'epic_immune_fraction_z', 'epic_caf_fraction_z', 'epic_endothelial_fraction_z',
    'mcp_t_cells_z', 'mcp_cytotoxic_lymphocytes_z', 'mcp_nk_cells_z', 'mcp_endothelial_cells_z', 'mcp_fibroblasts_z',
]
plot = data[data['score'].isin(selected)].copy()
plot['score_label'] = plot['score'].str.replace('_z', '', regex=False).str.replace('epic_', 'EPIC ', regex=False).str.replace('mcp_', 'MCP ', regex=False).str.replace('_', ' ', regex=False)
matrix = plot.pivot_table(index='semantic_state_family', columns='score_label', values='cohen_d', aggfunc='mean').reindex([s for s in STATE_ORDER[:5] if s in plot['semantic_state_family'].unique()])

fig, ax = plt.subplots(figsize=(5.2, 3.1))
cmap = LinearSegmentedColormap.from_list('blue_white_red', ['#2F5D8C', '#F8F8F8', PALETTE['red']])
im = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', cmap=cmap, vmin=-1.2, vmax=1.2)
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(c, 12) for c in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [STATE_LABELS.get(i, i) for i in matrix.index])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, label='Cohen d')
ax.set_title('Deconvolution effect sizes')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s6c_paths = save_panel(fig, 'S6c_deconvolution_effect_heatmap')

**Caption (English).** Deconvolution effect sizes across selected cell-state axes. Rows are portrait groups, columns are EPIC or MCP-counter cell-state scores, and colours show Cohen d.

**图注（中文）.** 不同画像组在反卷积细胞状态轴上的效应大小。行是画像组，列是 EPIC 或 MCP-counter 分数，颜色表示 Cohen d。

**小朋友版解释.** 这张图看画像组是不是对应不同细胞成分。红色偏高，蓝色偏低。

## Main Fig. 4d. Independent support for portrait claims

In [ ]:
# Figure 4d. Standard horizontal bar plot: portrait-claim support.
data = read_table('figure_4_claim_support')
data = data.copy()
data['claim_label'] = data['claim_type'].map(CLAIM_LABELS).fillna(data['claim_label'])
data['claim_type'] = data['claim_type'].map(CLAIM_TYPE_LABELS).fillna(data['claim_type'])
plot_data = data.sort_values('partial_or_strong_rate')

fig, ax = plt.subplots(figsize=(4.0, 2.8))
y = np.arange(len(plot_data))
ax.barh(y, plot_data['partial_or_strong_rate'], color=PALETTE['green'])
ax.set_yticks(y, [wrap_label(label, 20) for label in plot_data['claim_label']])
ax.set_xlim(0, 1)
ax.set_xlabel('partial or strong support')
ax.set_title('Portrait claims')
clean_axis(ax, grid='x')
fig4d_paths = save_panel(fig, 'Figure_4d_horizontal_bar_claim_support')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_4d_horizontal_bar_claim_support -->
**Caption (English).** Textual portrait claims are checked against supporting analyses. The x axis shows the fraction of claims with partial or strong support, and each horizontal bar represents one class of portrait statement.

**图注（中文）。** 将文字画像中的判断与支持性分析进行核对。横轴表示获得部分或强支持的判断比例，每个横条代表一类画像陈述。

**小朋友版解释。** 模型写出一句话后，我们再检查它有没有证据。横条越长，说明这类话越常能找到数据支持。


# Evidence block 6. Whole-profile portraits are not fully reducible to local summaries

Claim tested here: local marker/pathway/deconvolution summaries are useful, but they do not fully reconstruct the portrait representation. This defines a boundary rather than dismissing traditional bioinformatics.

## Extended Data Fig. 1a. Cross-validated local and portrait comparisons

In [ ]:
# Extended Data 1a. Standard grouped bar plot: cross-validated R2.
data = read_table('extended_data_1_cv_comparison')
outcomes = ['immune_support', 'context_support', 'tumor_like_support', 'mixed_evidence_support', 'boundary_flag_count']
outcome_labels = ['immune', 'context', 'tumour-like', 'several-signal', 'boundary count']
pred_order = ['labels_only', 'local_parts_only', 'labels_plus_local_parts', 'labels_local_plus_portrait']
colors = [PALETTE['grey_light'], PALETTE['blue_light'], PALETTE['blue'], PALETTE['green']]

fig, ax = plt.subplots(figsize=(4.6, 2.9))
x = np.arange(len(outcomes))
width = 0.18
for i, pred in enumerate(pred_order):
    values = [data[(data['outcome'] == outcome) & (data['predictor_set'] == pred)]['cv_r2'].mean() for outcome in outcomes]
    label = data[data['predictor_set'] == pred]['predictor_label'].iloc[0]
    ax.bar(x + (i - 1.5) * width, values, width=width, color=colors[i], label=label)
ax.set_xticks(x, outcome_labels, rotation=35, ha='right')
ax.set_ylabel('cross-validated R2')
ax.set_title('Local evidence and portrait information')
ax.legend(loc='best')
clean_axis(ax)
ed1a_paths = save_panel(fig, 'Extended_Data_1a_grouped_bar_cv_r2')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Extended_Data_1a_grouped_bar_cv_r2 -->
**Caption (English).** Cross-validated models test whether portrait information adds explanatory value beyond simpler summaries. The x axis lists outcomes, the y axis is cross-validated R2, and colours indicate different predictor sets.

**图注（中文）。** 交叉验证模型检验分子画像是否比更简单的摘要提供额外解释力。横轴是不同结果变量，纵轴是交叉验证 R2，颜色表示不同预测变量组合。

**小朋友版解释。** 这里比赛几种信息谁更会解释结果。柱子越高，说明那组信息越有用；如果加入画像后更高，说明画像带来了额外信息。


## Extended Data Fig. 1b. Portrait-group reducibility from labels and local evidence

In [ ]:
# Extended Data 1b. Standard horizontal bar plot: portrait-group reducibility.
data = read_table('extended_data_1_portrait_reducibility')
plot_data = data.set_index('predictor_set').loc[['majority_baseline', 'labels_only', 'local_parts_only', 'labels_plus_local_parts']].reset_index()
colors = [PALETTE['grey_light'], PALETTE['blue_light'], PALETTE['blue'], PALETTE['green']]

fig, ax = plt.subplots(figsize=(3.2, 2.6))
ax.barh(plot_data['predictor_label'], plot_data['macro_f1'], color=colors)
ax.set_xlim(0, 0.55)
ax.set_xlabel('macro F1')
ax.set_title('Portrait-group reducibility')
clean_axis(ax, grid='x')
ed1b_paths = save_panel(fig, 'Extended_Data_1b_horizontal_bar_reducibility')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Extended_Data_1b_horizontal_bar_reducibility -->
**Caption (English).** Portrait-group reducibility is evaluated with simpler predictors. The x axis is macro F1, and each horizontal bar shows how well a simpler predictor set can reproduce portrait groups.

**图注（中文）。** 用更简单的预测变量评估画像家族是否可被还原。横轴是 macro F1，每个横条表示一组简单预测变量对画像家族的复现程度。

**小朋友版解释。** 这张图问一个简单问题：只用普通标签或局部信息，能不能猜出模型的画像分组？横条不够满，说明画像不是完全由这些简单信息决定的。


# Evidence block 7. Stress tests and shortcut controls

Claim tested here: the RNA portrait signal is not fully explained by simple source, tissue, platform or metadata shortcuts, but public-data quality and source structure remain explicit boundaries.

## Main Fig. 5a. Controlled profile mixing

In [ ]:
# Figure 5a. Standard line plot with bootstrap bands: controlled profile mixing.
data = read_table('figure_5_mixing')
plot_data = data[data['model'] == 'temperature_scaled_sgd'].copy()
colors = {'normal_immune': PALETTE['green'], 'tumor_immune': PALETTE['orange'], 'tumor_normal': PALETTE['red']}
labels = {'normal_immune': 'normal + immune', 'tumor_immune': 'tumour + immune', 'tumor_normal': 'tumour + normal'}

fig, ax = plt.subplots(figsize=(3.4, 3.2))
for design, group in plot_data.groupby('design'):
    group = group.sort_values('fraction_b')
    color = colors.get(design, PALETTE['grey'])
    ax.plot(group['fraction_b'], group['openworld_mixed_or_unsupported_rate'], marker='o', lw=1.2, ms=3.5, color=color, label=labels.get(design, design))
    ax.fill_between(
        group['fraction_b'].to_numpy(dtype=float),
        group['openworld_mixed_or_unsupported_rate_lo'].to_numpy(dtype=float),
        group['openworld_mixed_or_unsupported_rate_hi'].to_numpy(dtype=float),
        color=color,
        alpha=0.16,
        lw=0,
    )
ax.set_xlabel('mixing fraction')
ax.set_ylabel('several-signal or weak-evidence rate')
ax.set_ylim(0, 1.05)
ax.set_xlim(-0.03, 1.03)
ax.set_title('Controlled profile mixing')
ax.legend(loc='best')
clean_axis(ax)
fig5a_paths = save_panel(fig, 'Figure_5a_line_controlled_profile_mixing')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_5a_line_controlled_profile_mixing -->
**Caption (English).** Controlled mixtures test whether portrait status changes when two RNA profiles are blended. The x axis is the mixing fraction, the y axis is the rate of several-signal or weak-evidence calls, and shaded bands show uncertainty.

**图注（中文）。** 受控混合实验测试当两个 RNA 样本被混合时，分子画像状态是否随之变化。横轴是混合比例，纵轴是多个信号或证据较弱判断的比例，阴影带表示不确定性。

**小朋友版解释。** 把两份 RNA 像调颜料一样按不同比例混在一起。横轴表示第二份加了多少，纵轴表示模型有多常说“这里不止一个信号”或“证据不够清楚”。


## Supplementary Fig. 7b. Disease-label confidence across controlled mixtures

In [ ]:
# S7b. Predefined disease-label confidence across controlled mixtures.
data = read_table('s7_mixing')
plot_data = data[data['model'] == 'temperature_scaled_sgd'].copy()
colors = {'normal_immune': PALETTE['green'], 'tumor_immune': PALETTE['orange'], 'tumor_normal': PALETTE['red']}

fig, ax = plt.subplots(figsize=(3.8, 3.0))
for design, g in plot_data.groupby('design'):
    g = g.sort_values('fraction_b')
    color = colors.get(design, PALETTE['grey'])
    ax.plot(g['fraction_b'], g['mean_closed_set_confidence'], marker='o', color=color, label=design.replace('_', ' + '))
    ax.fill_between(g['fraction_b'].to_numpy(), g['mean_closed_set_confidence_lo'].to_numpy(), g['mean_closed_set_confidence_hi'].to_numpy(), color=color, alpha=0.14, lw=0)
ax.set_xlabel('mixing fraction')
ax.set_ylabel('mean predefined-label confidence')
ax.set_ylim(0, max(1.0, float(plot_data['mean_closed_set_confidence_hi'].max()) * 1.05))
ax.set_title('Predefined-label confidence')
ax.legend(loc='best')
clean_axis(ax)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s7b_paths = save_panel(fig, 'S7b_mixing_predefined_label_confidence')

**Caption (English).** Predefined disease-label confidence across controlled mixtures. The x axis shows mixing fraction and the y axis shows mean predefined-label confidence with bootstrap intervals.

**图注（中文）.** 受控混合样本中的预设疾病标签置信度。横坐标是混合比例，纵坐标是预设标签平均置信度，阴影表示 bootstrap 区间。

**小朋友版解释.** 这张图看传统单标签模型在样本被混合后还会不会很自信。

## Main Fig. 5b. Metadata association

In [ ]:
# Figure 5b. Standard horizontal bar plot: metadata association.
data = read_table('figure_5_metadata_association')
plot_data = data[data['target'] == 'semantic_state_family'].copy()
plot_data = plot_data[plot_data['factor'].isin(['pool', 'project_prefix', 'project', 'expected_site_family', 'expected_disease_family', 'closed_set_disease_family'])]
plot_data['label'] = plot_data['factor_label'].replace({
    'external pool': 'pool',
    'source prefix': 'source prefix',
    'project/source': 'project',
    'expected tissue/site': 'tissue',
    'expected disease label': 'expected disease',
    'predefined disease label': 'predefined disease',
})
plot_data = plot_data.sort_values('nmi')
colors = [PALETTE['red_light'] if label == 'project' else PALETTE['grey'] for label in plot_data['label']]

fig, ax = plt.subplots(figsize=(3.2, 2.8))
ax.barh(plot_data['label'], plot_data['nmi'], color=colors)
ax.set_xlabel('NMI with portraits')
ax.set_title('Simple metadata links')
clean_axis(ax, grid='x')
fig5b_paths = save_panel(fig, 'Figure_5b_horizontal_bar_metadata_links')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_5b_horizontal_bar_metadata_links -->
**Caption (English).** Simple metadata factors are tested for association with portrait families. The x axis is normalized mutual information with portraits; longer bars indicate stronger association between a metadata field and the portrait assignment.

**图注（中文）。** 检验简单 metadata 因素与分子画像家族的关联。横轴是与画像之间的归一化互信息；横条越长，说明某个 metadata 字段与画像分配越相关。

**小朋友版解释。** 这里检查模型是不是可能在看“样本来自哪个项目、哪个组织、哪个标签”这些外部信息。横条越长，说明这种外部信息和画像越有关。


## Supplementary Fig. 8a. Metadata association with portrait outputs

In [ ]:
# S8a. Metadata association with portrait targets.
data = read_table('s8_metadata')
plot_data = data.copy()
plot_data['factor_short'] = plot_data['factor_label'].replace({
    'external pool': 'pool', 'source prefix': 'source prefix', 'project/source': 'project',
    'expected tissue/site': 'tissue', 'expected disease label': 'expected disease', 'fixed disease label': 'predefined disease',
})
fig, ax = plt.subplots(figsize=(4.8, 3.0))
factors = list(plot_data['factor_short'].drop_duplicates())
targets = list(plot_data['target_label'].drop_duplicates())
x = np.arange(len(factors))
width = 0.8 / max(1, len(targets))
for i, target in enumerate(targets):
    sub = plot_data[plot_data['target_label'] == target].set_index('factor_short').reindex(factors)
    ax.bar(x + (i - (len(targets)-1)/2)*width, sub['nmi'], width=width, label=target, color=[PALETTE['blue'], PALETTE['orange'], PALETTE['green']][i % 3])
ax.set_xticks(x, [wrap_label(f, 12) for f in factors], rotation=35, ha='right')
ax.set_ylabel('normalized mutual information')
ax.set_title('Metadata association with portrait outputs')
ax.legend(loc='best')
clean_axis(ax)
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s8a_paths = save_panel(fig, 'S8a_metadata_association_nmi')

**Caption (English).** Association between metadata factors and portrait outputs. The x axis lists metadata factors and the y axis shows normalized mutual information for each portrait target.

**图注（中文）.** metadata 因素与画像输出之间的关联。横坐标是不同 metadata 因素，纵坐标是 normalized mutual information。

**小朋友版解释.** 这张图检查模型是不是只是在偷看样本来源、组织或疾病名字。柱子越高，关联越强。

## Supplementary Fig. 8c. Within-label evidence separation

In [ ]:
# S8c. Within-label evidence separation.
data = read_table('s8_within_label').sort_values('weighted_silhouette_lift')
fig, ax = plt.subplots(figsize=(3.8, 2.8))
ax.barh(data['factor_label'], data['weighted_silhouette_lift'], color=PALETTE['violet'])
ax.axvline(0, color=PALETTE['black'], lw=0.7)
ax.set_xlabel('silhouette lift over permutation')
ax.set_title('Within-label evidence separation')
clean_axis(ax, grid='x')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s8c_paths = save_panel(fig, 'S8c_within_label_evidence_separation')

**Caption (English).** Within-label evidence separation. The x axis shows silhouette lift over permutation and the y axis lists label factors tested for within-label portrait separation.

**图注（中文）.** 同一标签内部的证据分离。横坐标是相对随机排列的 silhouette 提升，纵坐标是测试的标签因素。

**小朋友版解释.** 这张图看同一个疾病名字里面，画像能不能分出不同小组。越往右，说明分得更清楚。

## Main Fig. 5c. Residual signal after source and metadata controls

In [ ]:
# Figure 5c. Standard heatmap: incremental R2 after source controls.
data = read_table('figure_5_partial_r2_matrix')
matrix = data.set_index('control_label')

fig, ax = plt.subplots(figsize=(4.4, 3.1))
cmap = LinearSegmentedColormap.from_list('white_blue', ['#FFFFFF', PALETTE['blue_soft'], PALETTE['blue']])
im = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', cmap=cmap, vmin=0, vmax=0.22)
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(c, 10) for c in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [wrap_label(i, 16) for i in matrix.index])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.colorbar(im, ax=ax, label='incremental R2')
ax.set_title('After source controls')
fig5c_paths = save_panel(fig, 'Figure_5c_heatmap_source_controls')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_5c_heatmap_source_controls -->
**Caption (English).** Portrait signals are re-evaluated after controlling for source-related structure. Rows indicate control settings, columns indicate portrait-related outcomes, and colour shows the incremental R2 that remains after each control.

**图注（中文）。** 在控制数据来源相关结构后重新评估画像信号。行表示不同控制条件，列表示画像相关结果，颜色表示在相应控制后仍然保留的增量 R2。

**小朋友版解释。** 这张图是在问：如果先把“来自哪里”这些影响扣掉，画像还剩多少有用信息？颜色越深，说明扣掉之后仍然有信号。


## Main Fig. 5d. Robustness across quality subsets

In [ ]:
# Figure 5d. Standard horizontal bar plot: robustness subsets.
data = read_table('figure_5_robustness')
plot_data = data[data['subset_id'].isin([
    'all_external',
    'exclude_lowest_10pct_coverage',
    'high_coverage_top50',
    'low_coverage_bottom50',
    'external_180_only',
    'multisource_450_only',
    'source_site_disease',
])].copy()
if plot_data.empty:
    plot_data = data.head(7).copy()
plot_data = plot_data.head(7).iloc[::-1]

fig, ax = plt.subplots(figsize=(3.4, 3.1))
ax.barh(plot_data['subset_label'].map(lambda x: wrap_label(x, 24)), plot_data['mean_incremental_r2'], color=PALETTE['blue'])
ax.set_xlabel('mean incremental R2')
ax.set_title('Robustness subsets')
clean_axis(ax, grid='x')
fig5d_paths = save_panel(fig, 'Figure_5d_horizontal_bar_robustness_subsets')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_5d_horizontal_bar_robustness_subsets -->
**Caption (English).** Robustness is assessed across analysis subsets. The y axis lists sample subsets or filtering strategies, and the x axis shows the mean incremental R2 retained in each subset.

**图注（中文）。** 在不同分析子集上评估结果稳健性。纵轴列出不同样本子集或过滤策略，横轴表示每个子集中保留的平均增量 R2。

**小朋友版解释。** 我们把数据换成不同小组再看一次。横条还在，说明结论不是只靠某一小堆样本才出现。


## Supplementary Fig. 9a. Robustness subset analysis

In [ ]:
# S9a. Robustness subsets.
data = read_table('s9_robustness')
plot_data = data.sort_values('mean_incremental_r2')
fig, ax = plt.subplots(figsize=(4.2, 3.5))
ax.barh(plot_data['subset_label'].map(lambda x: wrap_label(x, 25)), plot_data['mean_incremental_r2'], color=PALETTE['blue'])
ax.set_xlabel('mean incremental R2')
ax.set_title('Residual signal across robustness subsets')
clean_axis(ax, grid='x')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s9a_paths = save_panel(fig, 'S9a_robustness_subset_incremental_r2')

**Caption (English).** Residual signal across robustness subsets. The x axis shows mean incremental R2 and the y axis lists data-quality or source subsets.

**图注（中文）.** 不同稳健性子集中的残余画像信号。横坐标是平均额外 R2，纵坐标是不同数据质量或来源子集。

**小朋友版解释.** 这张图看换一批样本、去掉质量差的样本后，画像信号还在不在。条越长，说明还在。

## Supplementary Fig. 9b. Quality metrics by portrait group

In [ ]:
# S9b. Quality metrics by portrait group.
data = read_table('s2_quality').sort_values('mean_gene_coverage')
fig, ax = plt.subplots(figsize=(4.2, 3.2))
y = np.arange(len(data))
ax.barh(y - 0.18, data['mean_gene_coverage'], height=0.35, color=PALETTE['blue_light'], label='mean gene coverage')
ax.barh(y + 0.18, data['epic_converged_rate'], height=0.35, color=PALETTE['green_light'], label='EPIC converged')
ax.set_yticks(y, portrait_display_labels(data['label'], 22))
ax.set_xlim(0, 1.0)
ax.set_xlabel('fraction or mean coverage')
ax.set_title('Quality metrics by portrait group')
ax.legend(loc='best')
clean_axis(ax, grid='x')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s9b_paths = save_panel(fig, 'S9b_quality_metrics_by_portrait')

**Caption (English).** Quality metrics by portrait group. The x axis shows mean gene coverage or EPIC convergence rate and the y axis lists portrait groups.

**图注（中文）.** 不同画像组的数据质量指标。横坐标是平均基因覆盖度或 EPIC 收敛率，纵坐标是画像组。

**小朋友版解释.** 这张图检查每类画像的数据是不是够完整、计算是不是稳定。

## Main Fig. 5e. Reliability-boundary flags

In [ ]:
# Figure 5e. Standard horizontal bar plot: reliability boundaries.
data = read_table('figure_5_boundaries')
plot_data = data.sort_values('fraction_samples', ascending=True)
colors = [PALETTE['red_light'] if value >= 0.28 else PALETTE['grey'] for value in plot_data['fraction_samples']]

fig, ax = plt.subplots(figsize=(3.8, 3.0))
ax.barh(plot_data['boundary_label'].map(lambda x: wrap_label(x, 18)), plot_data['fraction_samples'], color=colors)
ax.set_xlim(0, max(0.62, float(plot_data['fraction_samples'].max()) * 1.15))
ax.set_xlabel('fraction of external profiles')
ax.set_title('Reliability boundaries')
clean_axis(ax, grid='x')
fig5e_paths = save_panel(fig, 'Figure_5e_horizontal_bar_reliability_boundaries')

<!-- RNA_PORTRAIT_MAIN_CAPTION:Figure_5e_horizontal_bar_reliability_boundaries -->
**Caption (English).** Reliability-boundary flags summarize where interpretation should be cautious. The x axis is the fraction of external profiles affected by each boundary flag; red-tinted bars mark more frequent limitations.

**图注（中文）。** 可靠性边界标记总结哪些地方需要谨慎解释。横轴是受每类边界标记影响的外部样本比例；偏红的横条表示更常见的限制因素。

**小朋友版解释。** 这张图像模型的“注意事项清单”。横条越长，表示越多样本遇到这种需要小心解释的问题。


## Supplementary Fig. 10b. Reliability tiers by portrait group

In [ ]:
# S10b. Reliability tiers by portrait group.
data = read_table('s10_reliability')
tiers = [
    ('tier_A_high_confidence_fraction', 'high confidence', PALETTE['green']),
    ('tier_B_mixed_but_auditable_fraction', 'auditable mixed', PALETTE['blue_light']),
    ('tier_B_caution_boundary_fraction', 'caution boundary', PALETTE['orange_light']),
    ('tier_C_quality_limited_fraction', 'quality limited', PALETTE['grey_light']),
    ('tier_C_weak_or_unsupported_fraction', 'weak/unsupported', PALETTE['red_light']),
]
plot_data = data.sort_values('tier_A_high_confidence_fraction')
y = np.arange(len(plot_data))
left = np.zeros(len(plot_data))
fig, ax = plt.subplots(figsize=(4.8, 3.2))
for col, label, color in tiers:
    ax.barh(y, plot_data[col], left=left, color=color, label=label)
    left += plot_data[col].to_numpy()
ax.set_yticks(y, portrait_display_labels(plot_data['portrait_label'], 22))
ax.set_xlim(0, 1)
ax.set_xlabel('fraction of profiles')
ax.set_title('Reliability tiers by portrait group')
ax.legend(loc='best', fontsize=8)
clean_axis(ax, grid='x')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s10b_paths = save_panel(fig, 'S10b_reliability_tiers_by_portrait')

**Caption (English).** Reliability tiers by portrait group. Each horizontal stacked bar shows the fraction of profiles in high-confidence, auditable, cautionary or weak tiers.

**图注（中文）.** 不同画像组的可靠性等级。每个横向堆叠条表示高可信、可审计、需谨慎或弱证据样本的比例。

**小朋友版解释.** 这张图给每类画像分等级：哪些比较可靠，哪些要小心看。

## Supplementary Fig. 10c. Boundary flags by portrait group

In [ ]:
# S10c. Boundary flag fractions by portrait group.
data = read_table('s10_reliability')
flag_cols = [c for c in data.columns if c.startswith('flag_') and c.endswith('_fraction')]
plot_data = data.copy()
plot_data['portrait_label_display'] = plot_data['portrait_label'].map(portrait_display_label)
matrix = plot_data.set_index('portrait_label_display')[flag_cols]
matrix.columns = [c.replace('flag_', '').replace('_fraction', '').replace('_', ' ') for c in matrix.columns]
fig, ax = plt.subplots(figsize=(5.2, 3.4))
im = ax.imshow(matrix.to_numpy(dtype=float), aspect='auto', cmap=LinearSegmentedColormap.from_list('white_red', ['#FFFFFF', PALETTE['red_light'], PALETTE['red']]), vmin=0, vmax=max(0.75, float(matrix.max().max())))
ax.set_xticks(np.arange(matrix.shape[1]), [wrap_label(c, 12) for c in matrix.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(matrix.shape[0]), [wrap_label(x, 22) for x in matrix.index])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, label='fraction')
ax.set_title('Boundary flags by portrait group')
apply_font_sizes(fig, title_fontsize=TITLE_FONTSIZE, label_fontsize=LABEL_FONTSIZE, tick_fontsize=TICK_FONTSIZE, legend_fontsize=LEGEND_FONTSIZE)
s10c_paths = save_panel(fig, 'S10c_boundary_flags_by_portrait_heatmap')

**Caption (English).** Boundary flag fractions by portrait group. Rows are portrait groups, columns are boundary flags, and colours show the fraction of profiles carrying each flag.

**图注（中文）.** 不同画像组中的边界标记比例。行是画像组，列是边界标记，颜色表示带有该标记的样本比例。

**小朋友版解释.** 这张图看每类画像最容易遇到哪种问题。颜色越深，说明这个问题越常出现。

# Coverage check

Displayed result panels: Main Fig. 2a-d, Main Fig. 3a-d, Main Fig. 4a-d, Main Fig. 5a-e, Extended Data Fig. 1a-b, Supplementary Fig. 1a-f, 2a-c, 3a-c, 4a-d, 5a-c, 6a-c, 7b, 8a/c, 9a/b, 10b/c and 11a-d. Figure 1, the methods flowchart and composite figure pages are intentionally excluded.